In [2]:
pip install requests pandas numpy astropy lightkurve

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 85.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 123.3 MB/s  0:00:00
  Created wheel for fbpca: filename=fbpca-1.0-py3-none-any.whl size=11429 sha256=1ebf6edeee6312cf64aeaa91bb97a300e9b4148c7e22c2224d612d3c7a1f4544
  Stored in directory: /root/.cache/pip/wheels/30/85/d4/289ba063a9ae0ba3189c6914e992722575868b2cb41100401b
  Created wheel for memoization: filename=memoization-0.4.0-py3-none-any.whl size=50537 sha256=8305bc9748b8a60d3f775e34508ba9d031a6508ec941a359869b89fbfd2d6a27
  Stored in directory: /root/.cache/pip/wheels/5c/ef/65/c51178502ccdadab909ffa9339a9a89701

In [4]:
#!/usr/bin/env python3
"""
Phase 3: Deep Multi-Mission Forensic Pipeline for TIC 397073349 (TYC 4517-88-1)
Gaia DR3 555926586677752960

Performs:
  1. Complete Gaia DR3 parameter dump (gaia_source + astrophysical_parameters + nss_two_body_orbit)
  2. 60-arcsec Gaia DR3 neighborhood crowding & third-light contamination audit
  3. All-sector NASA TESS download (SPOC / TESS-SPOC / QLP), FITS CROWDSAP extraction
  4. High-precision multi-year TESS orbital period refinement + sector-by-sector stability
     of 1f_orb (beaming / reflection / spot) vs 2f_orb (ellipsoidal) amplitude and phase
  5. Exact Gaia-to-TESS RV-phase alignment check (verifies whether 1f_orb peaks at RV maximum,
     the smoking gun of relativistic Doppler beaming!)
  6. Multi-wavelength archival SED query (Gaia + 2MASS + AllWISE + GALEX) via VizieR
  7. Ellipsoidal & mass-function physical parameter space grid (dark vs luminous companion)

Outputs:
  - tic397073349_all_tess.csv
  - tic397073349_sector_stability.csv
  - tic397073349_phase3_diagnostics.png
  - Full telemetry summary in console -> paste back for Phase 4 modeling & paper drafting
"""

import io
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from astropy.timeseries import LombScargle
import lightkurve as lk

warnings.filterwarnings("ignore")

TARGET_GAIA_ID = 555926586677752960
TARGET_TIC = 397073349
TARGET_RA = 54.61398
TARGET_DEC = 79.34350

GAIA_TAP_ENDPOINTS = [
    ("ESA Gaia DR3 TAP", "https://gea.esac.esa.int/tap-server/tap/sync"),
    ("AIP Gaia DR3 Mirror", "https://gaia.aip.de/tap/sync"),
]

VIZIER_ASU_ENDPOINTS = [
    ("VizieR CfA Harvard", "https://vizier.cfa.harvard.edu/viz-bin/asu-tsv"),
    ("VizieR CDS Strasbourg", "https://vizier.cds.unistra.fr/viz-bin/asu-tsv"),
]

PIPELINE_PRIORITY = {"SPOC": 0, "TESS-SPOC": 1, "QLP": 2, "GSFC-ELEANOR-LITE": 3}


def run_tap_csv(endpoints, query: str, timeout: int = 90) -> pd.DataFrame:
    payload = {
        "REQUEST": "doQuery",
        "LANG": "ADQL",
        "FORMAT": "csv",
        "QUERY": query.strip(),
    }
    headers = {"User-Agent": "Mozilla/5.0 (AstrophysicsPipeline/3.0)"}
    last_err = None
    for name, url in endpoints:
        try:
            r = requests.post(url, data=payload, headers=headers, timeout=timeout)
            r.raise_for_status()
            txt = r.text.strip()
            if not txt or txt.startswith("<?xml") or txt.startswith("<html"):
                raise RuntimeError(f"Non-CSV response: {txt[:200]}")
            return pd.read_csv(io.StringIO(txt))
        except Exception as exc:
            last_err = exc
            print(f"[WARN] {name} failed: {exc}")
    raise RuntimeError(f"All TAP endpoints failed: {last_err}")


def fetch_full_gaia_profile():
    """Retrieve complete Gaia DR3 astrometric, spectroscopic, astrophysical, and NSS data."""
    q_target = f"""
    SELECT
        g.source_id, g.ra, g.dec, g.l, g.b,
        g.parallax, g.parallax_error, g.parallax_over_error,
        g.pmra, g.pmdec, g.ruwe, g.ipd_frac_multi_peak, g.non_single_star,
        g.phot_g_mean_mag, g.phot_bp_mean_mag, g.phot_rp_mean_mag, g.bp_rp,
        g.phot_bp_rp_excess_factor,
        g.radial_velocity, g.radial_velocity_error, g.rv_nb_transits,
        g.rv_expected_sig_to_noise, g.rv_template_teff, g.rv_template_logg, g.rv_template_fe_h,
        g.vbroad, g.vbroad_error, g.vbroad_nb_transits,
        g.teff_gspphot, g.logg_gspphot, g.mh_gspphot, g.ag_gspphot, g.ebpminrp_gspphot,
        a.radius_gspphot, a.radius_flame, a.lum_flame, a.mass_flame, a.age_flame,
        a.teff_gspspec, a.logg_gspspec, a.mh_gspspec,
        a.activityindex_espcs, a.activityindex_espcs_uncertainty,
        n.nss_solution_type, n.period, n.period_error,
        n.t_periastron, n.t_periastron_error,
        n.eccentricity, n.eccentricity_error,
        n.center_of_mass_velocity, n.center_of_mass_velocity_error,
        n.semi_amplitude_primary, n.semi_amplitude_primary_error,
        n.arg_periastron, n.arg_periastron_error,
        n.significance, n.goodness_of_fit, n.efficiency, n.flags
    FROM gaiadr3.gaia_source AS g
    LEFT JOIN gaiadr3.astrophysical_parameters AS a ON g.source_id = a.source_id
    LEFT JOIN gaiadr3.nss_two_body_orbit AS n ON g.source_id = n.source_id
    WHERE g.source_id = {TARGET_GAIA_ID}
    """
    df_t = run_tap_csv(GAIA_TAP_ENDPOINTS, q_target)

    # 60-arcsec neighborhood crowding check (TESS pixel = 21", typical aperture ~ 2-3 pix)
    q_neighbors = f"""
    SELECT
        source_id, ra, dec, phot_g_mean_mag, phot_rp_mean_mag, parallax, ruwe,
        DISTANCE(POINT('ICRS', ra, dec), POINT('ICRS', {TARGET_RA}, {TARGET_DEC})) * 3600.0 AS sep_arcsec
    FROM gaiadr3.gaia_source
    WHERE 1 = CONTAINS(
        POINT('ICRS', ra, dec),
        CIRCLE('ICRS', {TARGET_RA}, {TARGET_DEC}, {60.0 / 3600.0})
    )
    ORDER BY sep_arcsec ASC
    """
    df_nb = run_tap_csv(GAIA_TAP_ENDPOINTS, q_neighbors)
    return df_t.iloc[0], df_nb


def query_vizier_catalog(catalog: str, ra: float, dec: float, radius_arcsec: float = 5.0):
    """Query a VizieR catalog via ASU TSV with CfA/CDS failover."""
    params = {
        "-source": catalog,
        "-c": f"{ra:.6f} {dec:+.6f}",
        "-c.rs": f"{radius_arcsec:.1f}",
        "-out.max": "5",
        "-out.all": "1",
    }
    headers = {"User-Agent": "Mozilla/5.0 (AstrophysicsPipeline/3.0)"}
    for name, url in VIZIER_ASU_ENDPOINTS:
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            r.raise_for_status()
            lines = [ln for ln in r.text.splitlines() if ln and not ln.startswith("#")]
            # VizieR TSV has header, units, dashed line, then data rows
            if len(lines) >= 4:
                header = lines[0].split("\t")
                # Data starts after dashed separator line
                sep_idx = next((i for i, ln in enumerate(lines) if ln.startswith("---")), 2)
                data_lines = lines[sep_idx + 1:]
                if data_lines:
                    rows = [ln.split("\t") for ln in data_lines]
                    return pd.DataFrame(rows, columns=header)
            return pd.DataFrame()
        except Exception:
            continue
    return pd.DataFrame()


def fetch_all_tess_sectors(tic_id: int):
    """Download ALL unique TESS sectors for TIC 397073349 using highest-priority pipeline per sector."""
    sr = lk.search_lightcurve(f"TIC {tic_id}", mission="TESS")
    if sr is None or len(sr) == 0:
        raise RuntimeError(f"No TESS light curves found for TIC {tic_id}")

    tbl = sr.table
    print(f"\n[INFO] Total MAST light-curve entries found for TIC {tic_id}: {len(tbl)}")

    sector_best = {}
    for i in range(len(tbl)):
        author = str(tbl["author"][i]).upper()
        if author not in PIPELINE_PRIORITY:
            continue
        mission_str = str(tbl["mission"][i])
        prio = PIPELINE_PRIORITY[author]
        exptime = float(tbl["exptime"][i]) if "exptime" in tbl.colnames else 1800.0
        rank = (prio, exptime)
        if mission_str not in sector_best or rank < sector_best[mission_str][0]:
            sector_best[mission_str] = (rank, i, author, exptime)

    # Sort sectors numerically by sector number
    def sec_num(m_str):
        digits = "".join(ch for ch in m_str if ch.isdigit())
        return int(digits) if digits else 999

    selected = sorted(sector_best.items(), key=lambda kv: sec_num(kv[0]))
    print(f"[INFO] Unique TESS sectors available: {len(selected)}")

    sectors_data = []
    for sec_name, (_, idx, author, exptime) in selected:
        try:
            lc = sr[idx].download()
            if lc is None:
                continue
            meta = lc.meta if hasattr(lc, "meta") else {}
            crowdsap = meta.get("CROWDSAP", np.nan)
            flfrcsap = meta.get("FLFRCSAP", np.nan)
            sec_id = meta.get("SECTOR", sec_num(sec_name))

            lc = lc.remove_nans()
            t = np.asarray(lc.time.value, dtype=float)
            f = np.asarray(lc.flux.value, dtype=float)
            if lc.flux_err is not None and np.all(np.isfinite(lc.flux_err.value)):
                e = np.asarray(lc.flux_err.value, dtype=float)
            else:
                e = np.full_like(f, np.nanstd(f))

            good = np.isfinite(t) & np.isfinite(f) & (f > 0)
            t, f, e = t[good], f[good], e[good]
            if len(t) < 50:
                continue

            med = np.nanmedian(f)
            f_norm = f / med - 1.0
            e_norm = e / med

            # Sigma-clip outliers (> 5 sigma MAD)
            mad = 1.4826 * np.nanmedian(np.abs(f_norm))
            if mad > 0:
                keep = np.abs(f_norm) < 5.0 * mad
                t, f_norm, e_norm = t[keep], f_norm[keep], e_norm[keep]

            # Remove slow per-sector polynomial drift (degree 2, timescale >> 2.63 d)
            t_c = t - np.mean(t)
            coef = np.polyfit(t_c, f_norm, 2)
            f_detr = f_norm - np.polyval(coef, t_c)

            sectors_data.append({
                "sector": int(sec_id),
                "author": author,
                "exptime_s": float(exptime),
                "crowdsap": float(crowdsap) if crowdsap is not None else np.nan,
                "flfrcsap": float(flfrcsap) if flfrcsap is not None else np.nan,
                "t": t,
                "f": f_detr,
                "e": e_norm,
                "t_mid": float(np.mean(t)),
                "npts": len(t),
            })
            print(
                f"  -> Sector {int(sec_id):2d} | {author:<9} | exp={exptime:6.0f}s | "
                f"N={len(t):5d} | CROWDSAP={crowdsap} | t_mid={np.mean(t):.1f}"
            )
        except Exception as exc:
            print(f"  -> Sector {sec_name} FAILED: {exc}")

    return sectors_data


def fit_harmonics_linear(t, f, f_orb, t0_ref, n_harm=4):
    """
    Fit f(t) = c0 + sum_{k=1..n_harm} [ a_k cos(2 pi k f_orb (t-t0)) + b_k sin(2 pi k f_orb (t-t0)) ]
    Returns amplitudes, phases, parameter uncertainties, and model curve.
    """
    dt = t - t0_ref
    cols = [np.ones_like(dt)]
    for k in range(1, n_harm + 1):
        ang = 2.0 * np.pi * k * f_orb * dt
        cols.append(np.cos(ang))
        cols.append(np.sin(ang))
    X = np.column_stack(cols)
    beta, _, _, _ = np.linalg.lstsq(X, f, rcond=None)
    resid = f - X @ beta
    dof = max(len(t) - X.shape[1], 1)
    s2 = np.sum(resid ** 2) / dof
    cov = s2 * np.linalg.pinv(X.T @ X)
    errs = np.sqrt(np.maximum(np.diag(cov), 0.0))

    out = {"c0": beta[0], "rms_ppm": 1e6 * np.std(resid), "model": X @ beta, "resid": resid}
    for k in range(1, n_harm + 1):
        ak, bk = beta[2 * k - 1], beta[2 * k]
        sak, sbk = errs[2 * k - 1], errs[2 * k]
        amp = np.hypot(ak, bk)
        # Error propagation on amplitude
        amp_err = np.sqrt((ak * sak) ** 2 + (bk * sbk) ** 2) / max(amp, 1e-12)
        # Phase of maximum: a cos(x) + b sin(x) = A cos(x - phi_max), phi_max = atan2(b, a)
        phi_rad = np.arctan2(bk, ak) % (2.0 * np.pi)
        phi_cyc = phi_rad / (2.0 * np.pi)
        out[f"A{k}_ppm"] = 1e6 * amp
        out[f"A{k}_err_ppm"] = 1e6 * amp_err
        out[f"a{k}_ppm"] = 1e6 * ak
        out[f"b{k}_ppm"] = 1e6 * bk
        out[f"phi{k}_cyc"] = phi_cyc
    return out


def refine_orbital_period(t, f, p_init, t0_ref):
    """Non-linear least-squares refinement of orbital period using 3-harmonic model."""
    def residuals(params):
        p = params[0]
        f_orb = 1.0 / p
        dt = t - t0_ref
        cols = [np.ones_like(dt)]
        for k in (1, 2, 3):
            ang = 2.0 * np.pi * k * f_orb * dt
            cols.append(np.cos(ang))
            cols.append(np.sin(ang))
        X = np.column_stack(cols)
        beta, _, _, _ = np.linalg.lstsq(X, f, rcond=None)
        return f - X @ beta

    # Coarse grid search around p_init +/- 0.005 d to avoid local alias minima
    p_grid = np.linspace(p_init - 0.004, p_init + 0.004, 1601)
    rss = np.empty_like(p_grid)
    for i, pg in enumerate(p_grid):
        r = residuals([pg])
        rss[i] = np.dot(r, r)
    p_best_coarse = float(p_grid[int(np.argmin(rss))])

    res = least_squares(residuals, x0=[p_best_coarse], bounds=([p_best_coarse - 0.001], [p_best_coarse + 0.001]))
    p_opt = float(res.x[0])
    # Formal 1-sigma uncertainty from Jacobian
    dof = max(len(t) - 8, 1)
    s2 = np.sum(res.fun ** 2) / dof
    jtj = res.jac.T @ res.jac
    p_err = float(np.sqrt(s2 / max(jtj[0, 0], 1e-30)))
    return p_opt, p_err


def rv_phase_at_btjd(t_btjd, p_orb, t_peri_gaia, e, omega_deg):
    """
    Convert Gaia DR3 t_periastron (BJD - 2457388.5, i.e., J2016.0 = BJD 2457388.5)
    to TESS BTJD (BJD - 2457000.0):
      T_peri_BTJD = t_peri_gaia + 388.5
    Solve Kepler's equation and compute normalized radial velocity RV(t) ~ cos(nu + omega) + e cos(omega).
    Also returns conjunction epoch T_conj_BTJD where primary is closest to observer (nu + omega = 3*pi/2).
    """
    t_peri_btjd = t_peri_gaia + 388.5
    omega = np.radians(omega_deg)
    M = 2.0 * np.pi * ((t_btjd - t_peri_btjd) / p_orb) % (2.0 * np.pi)
    E = M.copy()
    for _ in range(12):
        E = E - (E - e * np.sin(E) - M) / (1.0 - e * np.cos(E))
    nu = 2.0 * np.arctan2(np.sqrt(1.0 + e) * np.sin(E / 2.0), np.sqrt(1.0 - e) * np.cos(E / 2.0))
    rv_unit = np.cos(nu + omega) + e * np.cos(omega)
    return rv_unit, nu


def main():
    print("=" * 100)
    print("PHASE 3: DEEP MULTI-MISSION FORENSICS FOR TIC 397073349 (TYC 4517-88-1)")
    print("=" * 100)

    # 1. Gaia DR3 Full Profile & Neighborhood
    print("\n[1/5] Fetching complete Gaia DR3 astrophysical, orbital & 60-arcsec neighborhood tables...")
    g_row, df_nb = fetch_full_gaia_profile()

    print("\n--- GAIA DR3 COMPLETE TARGET PROFILE ---")
    for col in g_row.index:
        val = g_row[col]
        if pd.notna(val):
            print(f"  {col:<32}: {val}")

    print("\n--- GAIA DR3 60-ARCSEC CROWDING AUDIT (TESS pixel = 21 arcsec) ---")
    g_target_mag = float(g_row["phot_g_mean_mag"])
    rp_target_mag = float(g_row["phot_rp_mean_mag"])
    df_nb["dG_mag"] = df_nb["phot_g_mean_mag"] - g_target_mag
    df_nb["dRP_mag"] = df_nb["phot_rp_mean_mag"] - rp_target_mag
    df_nb["flux_ratio_RP_pct"] = 100.0 * np.power(10.0, -0.4 * df_nb["dRP_mag"])
    print(df_nb[["source_id", "sep_arcsec", "phot_g_mean_mag", "phot_rp_mean_mag", "dRP_mag", "flux_ratio_RP_pct", "ruwe"]].head(12).to_string(index=False))

    total_contam_42 = df_nb.loc[(df_nb["sep_arcsec"] > 0.5) & (df_nb["sep_arcsec"] <= 42.0), "flux_ratio_RP_pct"].sum()
    print(f"  -> Sum of Gaia RP neighbor flux within 42\" (2 TESS pixels): {total_contam_42:.2f}% of target flux")

    # 2. Archival Multi-Wavelength SED Query (VizieR)
    print("\n[2/5] Querying archival photometry (2MASS, AllWISE, GALEX, Tycho-2, ROSAT/eROSITA)...")
    sed_catalogs = [
        ("2MASS (II/246/out)", "II/246/out", ["Jmag", "e_Jmag", "Hmag", "e_Hmag", "Kmag", "e_Kmag"]),
        ("AllWISE (II/328/allwise)", "II/328/allwise", ["W1mag", "e_W1mag", "W2mag", "e_W2mag", "W3mag", "e_W3mag", "W4mag"]),
        ("Tycho-2 (I/259/tyc2)", "I/259/tyc2", ["BTmag", "e_BTmag", "VTmag", "e_VTmag"]),
        ("APASS DR9 (II/336/apass9)", "II/336/apass9", ["Bmag", "e_Bmag", "Vmag", "e_Vmag", "g'mag", "r'mag", "i'mag"]),
        ("GALEX GR6+7 (II/335/galex_ais)", "II/335/galex_ais", ["FUVmag", "e_FUVmag", "NUVmag", "e_NUVmag"]),
        ("2RXS ROSAT (J/A+A/588/A103/cat2rxs)", "J/A+A/588/A103/cat2rxs", ["ExiML", "Cts", "CRate"]),
    ]
    print("\n--- MULTI-WAVELENGTH ARCHIVAL PHOTOMETRY ---")
    for label, cat, cols in sed_catalogs:
        rad = 30.0 if "ROSAT" in label else 6.0
        res = query_vizier_catalog(cat, TARGET_RA, TARGET_DEC, radius_arcsec=rad)
        if len(res) > 0:
            avail = [c for c in cols if c in res.columns]
            vals = {c: res.iloc[0][c] for c in avail}
            print(f"  {label:<36}: HIT -> {vals}")
        else:
            print(f"  {label:<36}: NO MATCH within {rad:.0f}\"")

    # 3. Download ALL TESS Sectors
    print("\n[3/5] Downloading ALL available NASA TESS sectors for TIC 397073349...")
    sectors = fetch_all_tess_sectors(TARGET_TIC)
    if not sectors:
        raise RuntimeError("No TESS sectors could be downloaded.")

    t_all = np.concatenate([s["t"] for s in sectors])
    f_all = np.concatenate([s["f"] for s in sectors])
    e_all = np.concatenate([s["e"] for s in sectors])
    sec_all = np.concatenate([np.full(s["npts"], s["sector"], dtype=int) for s in sectors])
    order = np.argsort(t_all)
    t_all, f_all, e_all, sec_all = t_all[order], f_all[order], e_all[order], sec_all[order]

    pd.DataFrame({"btjd": t_all, "flux_rel": f_all, "flux_err": e_all, "sector": sec_all}).to_csv(
        "tic397073349_all_tess.csv", index=False
    )

    p_gaia = float(g_row["period"])
    p_gaia_err = float(g_row["period_error"])
    tp_gaia = float(g_row["t_periastron"])
    ecc_gaia = float(g_row["eccentricity"])
    omega_gaia = float(g_row["arg_periastron"])
    k1_gaia = float(g_row["semi_amplitude_primary"])

    # Anchor phase 0.0 to Gaia periastron epoch in BTJD
    t_peri_btjd = tp_gaia + 388.5

    # 4. Refine Orbital Period across Multi-Year TESS Baseline & Check Harmonic Decomposition
    print("\n[4/5] Refining orbital period across full TESS baseline & running sector stability tests...")
    p_tess, p_tess_err = refine_orbital_period(t_all, f_all, p_gaia, t_peri_btjd)
    f_orb_tess = 1.0 / p_tess
    baseline_days = float(t_all.max() - t_all.min())

    print(f"\n--- ORBITAL PERIOD COMPARISON ---")
    print(f"  Total TESS sectors used : {len(sectors)} (Npts = {len(t_all):,}, baseline = {baseline_days:.1f} days)")
    print(f"  Gaia DR3 SB1 Period     : {p_gaia:.6f} +/- {p_gaia_err:.6f} d")
    print(f"  TESS Photometric Period : {p_tess:.7f} +/- {p_tess_err:.7f} d")
    print(f"  Delta P (TESS - Gaia)   : {(p_tess - p_gaia)*86400.0:+.2f} s ({(p_tess - p_gaia)/p_gaia_err:+.2f} sigma_Gaia)")

    # Global 4-harmonic fit
    h_glob = fit_harmonics_linear(t_all, f_all, f_orb_tess, t_peri_btjd, n_harm=4)
    print("\n--- GLOBAL HARMONIC DECOMPOSITION (Anchored to Gaia Periastron Phase = 0) ---")
    for k in range(1, 5):
        print(
            f"  Harmonic {k}f_orb: A{k} = {h_glob[f'A{k}_ppm']:7.1f} +/- {h_glob[f'A{k}_err_ppm']:5.1f} ppm | "
            f"phi_max = {h_glob[f'phi{k}_cyc']:.4f} cyc | "
            f"(a{k}={h_glob[f'a{k}_ppm']:+7.1f}, b{k}={h_glob[f'b{k}_ppm']:+7.1f} ppm)"
        )
    print(f"  Global residual RMS: {h_glob['rms_ppm']:.1f} ppm")

    # Test residual periodogram AFTER subtracting 4 harmonics at exact P_tess!
    freq_grid = np.linspace(0.05, 12.0, 60000)
    ls_res = LombScargle(t_all, h_glob["resid"])
    pow_res = ls_res.power(freq_grid)
    top_idx = np.argsort(pow_res)[::-1]
    print("\n--- TOP 5 RESIDUAL PERIODS AFTER SUBTRACTING EXACT TESS ORBITAL HARMONICS ---")
    reported = []
    med_pow = float(np.median(pow_res))
    for idx in top_idx:
        fp = float(freq_grid[idx])
        if all(abs(fp - r_f) > 0.03 for r_f in reported):
            reported.append(fp)
            pp = 1.0 / fp
            snr = float(pow_res[idx] / max(med_pow, 1e-12))
            amp_ppm = 1e6 * np.sqrt(2.0 * float(pow_res[idx]) * np.var(h_glob["resid"]))
            print(f"  #{len(reported)}: f = {fp:8.5f} d^-1 (P = {pp:7.4f} d, f/f_orb = {fp/f_orb_tess:6.3f}) | Amp ~ {amp_ppm:6.1f} ppm | PowSNR = {snr:6.1f}")
            if len(reported) >= 5:
                break

    # Sector-by-sector stability of 1f_orb and 2f_orb!
    print("\n--- SECTOR-BY-SECTOR STABILITY: 1f_orb (Beaming/Spot) vs 2f_orb (Ellipsoidal) ---")
    sec_rows = []
    hdr_s = f"{'Sec':>3} {'Pipe':<9} {'t_mid':>7} {'Npts':>5} {'A1(ppm)':>8} {'phi1':>6} {'A2(ppm)':>8} {'phi2':>6} {'A3(ppm)':>8} {'RMSppm':>7} {'CROWDSAP':>8}"
    print(hdr_s)
    print("-" * len(hdr_s))
    for s in sectors:
        hs = fit_harmonics_linear(s["t"], s["f"], f_orb_tess, t_peri_btjd, n_harm=3)
        sec_rows.append({
            "sector": s["sector"], "author": s["author"], "t_mid": s["t_mid"], "npts": s["npts"],
            "A1_ppm": hs["A1_ppm"], "A1_err": hs["A1_err_ppm"], "phi1": hs["phi1_cyc"],
            "A2_ppm": hs["A2_ppm"], "A2_err": hs["A2_err_ppm"], "phi2": hs["phi2_cyc"],
            "A3_ppm": hs["A3_ppm"], "rms_ppm": hs["rms_ppm"], "crowdsap": s["crowdsap"],
        })
        cs_str = f"{s['crowdsap']:8.4f}" if np.isfinite(s["crowdsap"]) else "     NaN"
        print(
            f"{s['sector']:>3d} {s['author']:<9} {s['t_mid']:>7.1f} {s['npts']:>5d} "
            f"{hs['A1_ppm']:>8.1f} {hs['phi1_cyc']:>6.3f} {hs['A2_ppm']:>8.1f} {hs['phi2_cyc']:>6.3f} "
            f"{hs['A3_ppm']:>8.1f} {hs['rms_ppm']:>7.1f} {cs_str}"
        )
    df_sec = pd.DataFrame(sec_rows)
    df_sec.to_csv("tic397073349_sector_stability.csv", index=False)

    # 5. RV-Phase vs Photometric-Phase Alignment Check (Doppler Beaming Test!)
    print("\n[5/5] Testing RV-to-Flux Phase Alignment (Relativistic Doppler Beaming vs Reflection)...")
    phase_grid = np.linspace(0.0, 1.0, 1000, endpoint=False)
    t_test = t_peri_btjd + phase_grid * p_tess
    rv_unit, _ = rv_phase_at_btjd(t_test, p_tess, tp_gaia, ecc_gaia, omega_gaia)
    # Note: In Gaia convention, positive RV = moving AWAY from observer (dimmer in Doppler beaming!)
    # So relativistic Doppler beaming flux peaks when RV is MOST NEGATIVE (primary approaching observer)!
    phi_rv_min = float(phase_grid[int(np.argmin(rv_unit))])  # max approaching velocity -> beaming peak!
    phi_rv_max = float(phase_grid[int(np.argmax(rv_unit))])  # max receding velocity -> beaming trough!

    # Compute binned phase curve on refined P_tess
    phase_obs = ((t_all - t_peri_btjd) / p_tess) % 1.0
    n_bins = 100
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    b_med = np.array([np.median(f_all[(phase_obs >= edges[i]) & (phase_obs < edges[i + 1])]) for i in range(n_bins)])
    b_err = np.array([
        1.4826 * np.median(np.abs(f_all[(phase_obs >= edges[i]) & (phase_obs < edges[i + 1])] - b_med[i]))
        / np.sqrt(max(np.sum((phase_obs >= edges[i]) & (phase_obs < edges[i + 1])), 1))
        for i in range(n_bins)
    ])

    # Evaluate 1st harmonic peak phase vs expected beaming peak phase
    # Note: phase propagation uncertainty from Gaia J2016.0 to TESS (~2020-2025, ~2500 days ~ 950 orbits):
    n_orbits_gap = (np.mean(t_all) - t_peri_btjd) / p_tess
    sigma_phi_prop = abs(n_orbits_gap) * (p_gaia_err / p_gaia)
    print(f"  Expected RV-approaching (Doppler beaming peak) phase : {phi_rv_min:.3f} cyc")
    print(f"  Expected RV-receding    (Doppler beaming min)  phase : {phi_rv_max:.3f} cyc")
    print(f"  Observed TESS 1f_orb peak phase                      : {h_glob['phi1_cyc']:.3f} cyc")
    print(f"  Phase propagation uncertainty from Gaia J2016.0      : +/- {sigma_phi_prop:.3f} cyc ({n_orbits_gap:.0f} orbits)")
    print(f"  Theoretical beaming semi-amplitude (K1={k1_gaia:.1f} km/s, alpha_beam~2.5): {1e6 * 2.5 * k1_gaia / 299792.458:.0f} ppm")
    print(f"  Observed global 1f_orb semi-amplitude A1             : {h_glob['A1_ppm']:.1f} +/- {h_glob['A1_err_ppm']:.1f} ppm")

    # Build 4-panel diagnostic figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 9.5), constrained_layout=True)

    # Panel A: Phase-folded all-sector light curve + harmonic fit + RV quadrature markers
    ax = axes[0, 0]
    ax.errorbar(centers, 1e6 * b_med, yerr=1e6 * b_err, fmt="o", ms=3.2, color="navy", ecolor="lightsteelblue", capsize=1.5, label="Binned TESS (all sectors)")
    t_mod = t_peri_btjd + phase_grid * p_tess
    # Reconstruct global 4-harmonic curve on phase_grid
    mod_vals = np.full_like(phase_grid, h_glob["c0"])
    for k in range(1, 5):
        ang = 2.0 * np.pi * k * phase_grid
        mod_vals += (h_glob[f"a{k}_ppm"] * 1e-6) * np.cos(ang) + (h_glob[f"b{k}_ppm"] * 1e-6) * np.sin(ang)
    ax.plot(phase_grid, 1e6 * mod_vals, color="crimson", lw=2.0, label="4-harmonic orbital model")
    ax.axvline(phi_rv_min, color="forestgreen", ls="--", lw=1.2, label=f"Max approach RV (phi={phi_rv_min:.2f})")
    ax.axvline(phi_rv_max, color="darkorange", ls=":", lw=1.2, label=f"Max recede RV (phi={phi_rv_max:.2f})")
    ax.set_xlabel("Orbital Phase (Gaia periastron = 0)", fontsize=10)
    ax.set_ylabel(r"Relative Flux $\Delta F / F$ (ppm)", fontsize=10)
    ax.set_title(f"TIC 397073349 (TYC 4517-88-1): {len(sectors)}-Sector TESS Phase Fold\nP_TESS = {p_tess:.7f} d | A2 = {h_glob['A2_ppm']:.0f} ppm, A1 = {h_glob['A1_ppm']:.0f} ppm", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.25)

    # Panel B: Residual Lomb-Scargle periodogram
    ax = axes[0, 1]
    amp_spec_ppm = 1e6 * np.sqrt(2.0 * pow_res * np.var(h_glob["resid"]))
    ax.plot(freq_grid, amp_spec_ppm, color="darkslateblue", lw=0.8)
    for k in range(1, 5):
        ax.axvline(k * f_orb_tess, color="crimson", ls="--", lw=0.8, alpha=0.6)
    ax.set_xlim(0.05, 5.0)
    ax.set_xlabel(r"Frequency ($\mathrm{d}^{-1}$)", fontsize=10)
    ax.set_ylabel("Residual Amplitude (ppm)", fontsize=10)
    ax.set_title("Residual Amplitude Spectrum (Post 4-Harmonic Subtraction)\nDashed red = 1..4 f_orb", fontsize=10, fontweight="bold")
    ax.grid(alpha=0.25)

    # Panel C: Sector-by-sector stability of A2 (ellipsoidal) & A1 (beaming/spot)
    ax = axes[1, 0]
    ax.errorbar(df_sec["t_mid"], df_sec["A2_ppm"], yerr=df_sec["A2_err"], fmt="s-", color="crimson", label="A2 (2f_orb Ellipsoidal)")
    ax.errorbar(df_sec["t_mid"], df_sec["A1_ppm"], yerr=df_sec["A1_err"], fmt="o-", color="teal", label="A1 (1f_orb Beaming/Spot)")
    ax.set_xlabel("Sector Midpoint Time (BTJD)", fontsize=10)
    ax.set_ylabel("Harmonic Semi-Amplitude (ppm)", fontsize=10)
    ax.set_title("Multi-Year Sector-by-Sector Amplitude Stability", fontsize=10, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)

    # Panel D: Sector-by-sector phase stability of phi2 & phi1
    ax = axes[1, 1]
    ax.plot(df_sec["t_mid"], df_sec["phi2"], "s-", color="crimson", label="phi2 (2f_orb phase)")
    ax.plot(df_sec["t_mid"], df_sec["phi1"], "o-", color="teal", label="phi1 (1f_orb phase)")
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Sector Midpoint Time (BTJD)", fontsize=10)
    ax.set_ylabel("Phase of Maximum (cycles)", fontsize=10)
    ax.set_title("Multi-Year Phase Stability (Constant = Tidal/Beaming; Drift = Spots)", fontsize=10, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)

    fig.savefig("tic397073349_phase3_diagnostics.png", dpi=180)
    plt.close(fig)
    print("\n[SAVED] tic397073349_all_tess.csv, tic397073349_sector_stability.csv, tic397073349_phase3_diagnostics.png")


if __name__ == "__main__":
    main()

PHASE 3: DEEP MULTI-MISSION FORENSICS FOR TIC 397073349 (TYC 4517-88-1)

[1/5] Fetching complete Gaia DR3 astrophysical, orbital & 60-arcsec neighborhood tables...

--- GAIA DR3 COMPLETE TARGET PROFILE ---
  source_id                       : 555926586677752960
  ra                              : 54.613980049872005
  dec                             : 79.34350268825605
  l                               : 130.4148698285385
  b                               : 18.98643784685197
  parallax                        : 1.687112001867416
  parallax_error                  : 0.011869555
  parallax_over_error             : 142.13776
  pmra                            : 12.262580160900765
  pmdec                           : -12.917226184700525
  ruwe                            : 0.79920894
  ipd_frac_multi_peak             : 0
  non_single_star                 : 2
  phot_g_mean_mag                 : 11.531674
  phot_bp_mean_mag                : 11.778843
  phot_rp_mean_mag                : 11.127526
  

In [5]:
#!/usr/bin/env python3
"""
Phase 4 (Memory-Safe): Complete Physical Modeling, Multi-Band SED Fit, Companion Mass
Constraints, and Publication-Grade Figure Generation for TYC 4517-88-1
(TIC 397073349 / Gaia DR3 555926586677752960).

Memory-safe upgrades for Google Colab (12.7 GB RAM):
  - Uses O(N log N) astropy.timeseries.LombScargle(method='fast') instead of SciPy 1.15+
    broadcast lombscargle (cuts periodogram RAM from ~25 GB down to <20 MB)
  - Uses scalar minimize_scalar for orbital period refinement (zero Jacobian allocation)
  - Auto-recovers TESS CSV via lightkurve if an OOM restart wiped /content/
  - Prints peak RSS memory usage at every stage

Outputs:
  - phase4_paper_numbers.txt         : Complete verified numerical report
  - tic397073349_sed_photometry.csv  : Archival multi-band photometry table
  - fig1_tess_phase_and_sectors.png  : Publication Fig 1 (8-sector phase fold + stability)
  - fig2_frequency_and_spots.png     : Publication Fig 2 (Residual spectrum + Hut pseudo-sync)
  - fig3_sed_and_hr_position.png     : Publication Fig 3 (Multi-band SED fit)
  - fig4_companion_constraints.png   : Publication Fig 4 (Inclination vs M2 constraints)
"""

import gc
import io
import os
import resource
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar

try:
    from astropy.timeseries import LombScargle as AstropyLS
    HAS_ASTROPY_LS = True
except Exception:
    HAS_ASTROPY_LS = False

warnings.filterwarnings("ignore")

TARGET_GAIA_ID = 555926586677752960
TARGET_TIC = 397073349
TARGET_RA = 54.61398
TARGET_DEC = 79.34350

GAIA_TAP_ENDPOINTS = [
    ("ESA Gaia DR3 TAP", "https://gea.esac.esa.int/tap-server/tap/sync"),
    ("AIP Gaia DR3 Mirror", "https://gaia.aip.de/tap/sync"),
]

VIZIER_ASU_ENDPOINTS = [
    ("VizieR CfA Harvard", "https://vizier.cfa.harvard.edu/viz-bin/asu-tsv"),
    ("VizieR CDS Strasbourg", "https://vizier.cds.unistra.fr/viz-bin/asu-tsv"),
]

PIPELINE_PRIORITY = {"SPOC": 0, "TESS-SPOC": 1, "QLP": 2, "GSFC-ELEANOR-LITE": 3}

# Physical constants (SI)
H_PLANCK = 6.62607015e-34   # J s
C_LIGHT = 2.99792458e8      # m s^-1
K_BOLTZ = 1.380649e-23      # J K^-1
R_SUN_M = 6.957e8           # m
PC_M = 3.085677581e16       # m


def peak_ram_mb() -> float:
    """Return current process peak resident memory in MB (Linux/Colab)."""
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0


def fast_lombscargle_amp_ppm(t: np.ndarray, y: np.ndarray, freqs: np.ndarray) -> np.ndarray:
    """
    Memory-safe Lomb-Scargle amplitude spectrum in ppm.
    Uses O(N log N) Press & Rybicki FFT via astropy (peak RAM <20 MB),
    with a strictly chunked pure-NumPy fallback if astropy is unavailable.
    """
    y0 = np.asarray(y, dtype=np.float64) - float(np.mean(y))
    t0 = np.asarray(t, dtype=np.float64)
    var_y = float(np.var(y0))
    if var_y <= 0.0:
        return np.zeros_like(freqs, dtype=np.float64)

    if HAS_ASTROPY_LS:
        ls = AstropyLS(t0, y0, center_data=True, fit_mean=True, normalization="standard")
        power = ls.power(freqs, method="fast", assume_regular_frequency=True)
        power = np.clip(np.asarray(power, dtype=np.float64), 0.0, 1.0)
        return 1e6 * np.sqrt(2.0 * power * var_y)

    # Strictly chunked pure-NumPy GLS fallback (chunk_size=128 -> <45 MB RAM)
    chunk_size = 128
    yy = float(np.dot(y0, y0))
    power = np.empty(len(freqs), dtype=np.float64)
    for start in range(0, len(freqs), chunk_size):
        end = min(start + chunk_size, len(freqs))
        w = 2.0 * np.pi * freqs[start:end]
        ang = np.outer(t0, w)  # shape (N, <=128)
        c = np.cos(ang)
        s = np.sin(ang)
        yc = y0 @ c
        ys = y0 @ s
        cc = np.sum(c * c, axis=0)
        ss = np.sum(s * s, axis=0)
        cs = np.sum(c * s, axis=0)
        det = np.maximum(cc * ss - cs * cs, 1e-18)
        p_chunk = (ss * yc * yc + cc * ys * ys - 2.0 * cs * yc * ys) / (det * yy)
        power[start:end] = np.clip(p_chunk, 0.0, 1.0)
        del ang, c, s
    return 1e6 * np.sqrt(2.0 * power * var_y)


def run_tap_csv(endpoints, query: str, timeout: int = 90) -> pd.DataFrame:
    payload = {"REQUEST": "doQuery", "LANG": "ADQL", "FORMAT": "csv", "QUERY": query.strip()}
    headers = {"User-Agent": "Mozilla/5.0 (AstrophysicsPipeline/4.1)"}
    last_err = None
    for _, url in endpoints:
        try:
            r = requests.post(url, data=payload, headers=headers, timeout=timeout)
            r.raise_for_status()
            txt = r.text.strip()
            if not txt or txt.startswith("<?xml") or txt.startswith("<html"):
                raise RuntimeError(f"Non-CSV response: {txt[:200]}")
            return pd.read_csv(io.StringIO(txt))
        except Exception as exc:
            last_err = exc
    raise RuntimeError(f"All TAP endpoints failed: {last_err}")


def query_vizier_catalog(catalog: str, ra: float, dec: float, radius_arcsec: float = 6.0) -> pd.DataFrame:
    params = {
        "-source": catalog,
        "-c": f"{ra:.6f} {dec:+.6f}",
        "-c.rs": f"{radius_arcsec:.1f}",
        "-out.max": "5",
        "-out.all": "1",
    }
    headers = {"User-Agent": "Mozilla/5.0 (AstrophysicsPipeline/4.1)"}
    for _, url in VIZIER_ASU_ENDPOINTS:
        try:
            r = requests.get(url, params=params, headers=headers, timeout=30)
            r.raise_for_status()
            lines = [ln for ln in r.text.splitlines() if ln and not ln.startswith("#")]
            if len(lines) >= 4:
                header = lines[0].split("\t")
                sep_idx = next((i for i, ln in enumerate(lines) if ln.startswith("---")), 2)
                data_lines = lines[sep_idx + 1:]
                if data_lines:
                    rows = [ln.split("\t") for ln in data_lines]
                    return pd.DataFrame(rows, columns=header)
            return pd.DataFrame()
        except Exception:
            continue
    return pd.DataFrame()


def ensure_tess_data():
    """
    Load local 'tic397073349_all_tess.csv' if present. If Colab reset wiped it,
    automatically re-download all TESS sectors via lightkurve and recreate it.
    Also loads CROWDSAP map from 'tic397073349_sector_stability.csv' if present.
    """
    crowdsap_map = {}
    if os.path.exists("tic397073349_sector_stability.csv"):
        tmp_s = pd.read_csv("tic397073349_sector_stability.csv")
        if "sector" in tmp_s.columns and "crowdsap" in tmp_s.columns:
            crowdsap_map = dict(zip(tmp_s["sector"].astype(int), tmp_s["crowdsap"]))

    if os.path.exists("tic397073349_all_tess.csv"):
        tess = pd.read_csv("tic397073349_all_tess.csv")
        return tess, crowdsap_map

    print("[INFO] 'tic397073349_all_tess.csv' not found on disk; re-downloading TESS sectors...")
    import lightkurve as lk
    sr = lk.search_lightcurve(f"TIC {TARGET_TIC}", mission="TESS")
    tbl = sr.table
    sector_best = {}
    for i in range(len(tbl)):
        author = str(tbl["author"][i]).upper()
        if author not in PIPELINE_PRIORITY:
            continue
        m_str = str(tbl["mission"][i])
        exptime = float(tbl["exptime"][i]) if "exptime" in tbl.colnames else 1800.0
        rank = (PIPELINE_PRIORITY[author], exptime)
        if m_str not in sector_best or rank < sector_best[m_str][0]:
            sector_best[m_str] = (rank, i, author)

    def sec_num(m_str):
        digits = "".join(ch for ch in m_str if ch.isdigit())
        return int(digits) if digits else 999

    rows = []
    for sec_name, (_, idx, _) in sorted(sector_best.items(), key=lambda kv: sec_num(kv[0])):
        try:
            lc = sr[idx].download()
            if lc is None:
                continue
            meta = lc.meta if hasattr(lc, "meta") else {}
            sec_id = int(meta.get("SECTOR", sec_num(sec_name)))
            if meta.get("CROWDSAP") is not None:
                crowdsap_map[sec_id] = float(meta["CROWDSAP"])
            lc = lc.remove_nans()
            t = np.asarray(lc.time.value, dtype=float)
            f = np.asarray(lc.flux.value, dtype=float)
            e = (
                np.asarray(lc.flux_err.value, dtype=float)
                if (lc.flux_err is not None and np.all(np.isfinite(lc.flux_err.value)))
                else np.full_like(f, np.nanstd(f))
            )
            good = np.isfinite(t) & np.isfinite(f) & (f > 0)
            t, f, e = t[good], f[good], e[good]
            med = np.nanmedian(f)
            f_norm = f / med - 1.0
            e_norm = e / med
            mad = 1.4826 * np.nanmedian(np.abs(f_norm))
            if mad > 0:
                keep = np.abs(f_norm) < 5.0 * mad
                t, f_norm, e_norm = t[keep], f_norm[keep], e_norm[keep]
            t_c = t - np.mean(t)
            f_detr = f_norm - np.polyval(np.polyfit(t_c, f_norm, 2), t_c)
            rows.append(pd.DataFrame({
                "btjd": t, "flux_rel": f_detr, "flux_err": e_norm,
                "sector": np.full(len(t), sec_id, dtype=int),
            }))
        except Exception:
            continue

    tess = pd.concat(rows, ignore_index=True).sort_values("btjd").reset_index(drop=True)
    tess.to_csv("tic397073349_all_tess.csv", index=False)
    return tess, crowdsap_map


def fetch_gaia_and_neighbors():
    q_t = f"""
    SELECT
        g.source_id, g.ra, g.dec, g.l, g.b,
        g.parallax, g.parallax_error, g.parallax_over_error,
        g.pmra, g.pmra_error, g.pmdec, g.pmdec_error,
        g.ruwe, g.ipd_frac_multi_peak, g.non_single_star,
        g.phot_g_mean_mag, g.phot_bp_mean_mag, g.phot_rp_mean_mag, g.bp_rp,
        g.phot_bp_rp_excess_factor,
        g.radial_velocity, g.radial_velocity_error, g.rv_nb_transits,
        g.rv_expected_sig_to_noise, g.rv_template_teff, g.rv_template_logg, g.rv_template_fe_h,
        g.vbroad, g.vbroad_error, g.vbroad_nb_transits,
        g.teff_gspphot, g.teff_gspphot_lower, g.teff_gspphot_upper,
        g.logg_gspphot, g.mh_gspphot, g.ag_gspphot, g.ebpminrp_gspphot,
        a.radius_gspphot, a.radius_flame, a.radius_flame_lower, a.radius_flame_upper,
        a.lum_flame, a.lum_flame_lower, a.lum_flame_upper,
        a.mass_flame, a.mass_flame_lower, a.mass_flame_upper,
        a.age_flame, a.teff_gspspec, a.logg_gspspec, a.mh_gspspec,
        a.activityindex_espcs, a.activityindex_espcs_uncertainty,
        n.nss_solution_type, n.period, n.period_error,
        n.t_periastron, n.t_periastron_error,
        n.eccentricity, n.eccentricity_error,
        n.center_of_mass_velocity, n.center_of_mass_velocity_error,
        n.semi_amplitude_primary, n.semi_amplitude_primary_error,
        n.arg_periastron, n.arg_periastron_error,
        n.significance, n.goodness_of_fit, n.efficiency, n.flags
    FROM gaiadr3.gaia_source AS g
    LEFT JOIN gaiadr3.astrophysical_parameters AS a ON g.source_id = a.source_id
    LEFT JOIN gaiadr3.nss_two_body_orbit AS n ON g.source_id = n.source_id
    WHERE g.source_id = {TARGET_GAIA_ID}
    """
    df_t = run_tap_csv(GAIA_TAP_ENDPOINTS, q_t)

    q_nb = f"""
    SELECT
        source_id, phot_g_mean_mag, phot_rp_mean_mag, parallax, ruwe,
        DISTANCE(POINT('ICRS', ra, dec), POINT('ICRS', {TARGET_RA}, {TARGET_DEC})) * 3600.0 AS sep_arcsec
    FROM gaiadr3.gaia_source
    WHERE 1 = CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {TARGET_RA}, {TARGET_DEC}, {60.0 / 3600.0}))
    ORDER BY sep_arcsec ASC
    """
    df_nb = run_tap_csv(GAIA_TAP_ENDPOINTS, q_nb)
    return df_t.iloc[0], df_nb


def safe_float(val, default=np.nan):
    try:
        s = str(val).strip()
        return float(s) if s != "" else default
    except Exception:
        return default


def build_sed_table(g_row):
    """Assemble multi-band archival SED from Gaia DR3, Tycho-2, APASS, 2MASS, AllWISE, GALEX."""
    bands = []

    def add_band(name, lam_um, fnu0_jy, k_av, mag, err):
        if np.isfinite(mag):
            e_use = err if (np.isfinite(err) and err > 0.01) else 0.03
            e_use = max(e_use, 0.025)
            bands.append({
                "band": name, "lam_um": lam_um, "fnu0_jy": fnu0_jy,
                "k_av": k_av, "mag": mag, "mag_err": e_use
            })

    add_band("Gaia_BP", 0.5110, 3552.0, 1.083, safe_float(g_row["phot_bp_mean_mag"]), 0.025)
    add_band("Gaia_G",  0.6218, 3229.0, 0.836, safe_float(g_row["phot_g_mean_mag"]), 0.025)
    add_band("Gaia_RP", 0.7769, 2555.0, 0.634, safe_float(g_row["phot_rp_mean_mag"]), 0.025)

    tyc = query_vizier_catalog("I/259/tyc2", TARGET_RA, TARGET_DEC, 5.0)
    if len(tyc) > 0:
        add_band("Tycho_BT", 0.4203, 4020.0, 1.37, safe_float(tyc.iloc[0].get("BTmag")), safe_float(tyc.iloc[0].get("e_BTmag")))
        add_band("Tycho_VT", 0.5319, 3690.0, 1.02, safe_float(tyc.iloc[0].get("VTmag")), safe_float(tyc.iloc[0].get("e_VTmag")))

    ap = query_vizier_catalog("II/336/apass9", TARGET_RA, TARGET_DEC, 5.0)
    if len(ap) > 0:
        add_band("APASS_B", 0.4361, 4063.0, 1.32, safe_float(ap.iloc[0].get("Bmag")), safe_float(ap.iloc[0].get("e_Bmag")))
        add_band("APASS_V", 0.5448, 3636.0, 1.00, safe_float(ap.iloc[0].get("Vmag")), safe_float(ap.iloc[0].get("e_Vmag")))
        add_band("APASS_g", 0.4640, 3631.0, 1.21, safe_float(ap.iloc[0].get("g'mag")), safe_float(ap.iloc[0].get("e_g'mag")))
        add_band("APASS_r", 0.6122, 3631.0, 0.87, safe_float(ap.iloc[0].get("r'mag")), safe_float(ap.iloc[0].get("e_r'mag")))
        add_band("APASS_i", 0.7439, 3631.0, 0.68, safe_float(ap.iloc[0].get("i'mag")), safe_float(ap.iloc[0].get("e_i'mag")))

    tm = query_vizier_catalog("II/246/out", TARGET_RA, TARGET_DEC, 5.0)
    if len(tm) > 0:
        add_band("2MASS_J",  1.2350, 1594.0, 0.282, safe_float(tm.iloc[0].get("Jmag")), safe_float(tm.iloc[0].get("e_Jmag")))
        add_band("2MASS_H",  1.6620, 1024.0, 0.175, safe_float(tm.iloc[0].get("Hmag")), safe_float(tm.iloc[0].get("e_Hmag")))
        add_band("2MASS_Ks", 2.1590,  666.7, 0.112, safe_float(tm.iloc[0].get("Kmag")), safe_float(tm.iloc[0].get("e_Kmag")))

    aw = query_vizier_catalog("II/328/allwise", TARGET_RA, TARGET_DEC, 5.0)
    if len(aw) > 0:
        add_band("WISE_W1", 3.3526, 309.54, 0.059, safe_float(aw.iloc[0].get("W1mag")), safe_float(aw.iloc[0].get("e_W1mag")))
        add_band("WISE_W2", 4.6028, 171.79, 0.044, safe_float(aw.iloc[0].get("W2mag")), safe_float(aw.iloc[0].get("e_W2mag")))
        w3m = safe_float(aw.iloc[0].get("W3mag"))
        w3e = safe_float(aw.iloc[0].get("e_W3mag"))
        if np.isfinite(w3m) and np.isfinite(w3e) and w3e < 0.35:
            add_band("WISE_W3", 11.5608, 31.674, 0.025, w3m, w3e)

    gx = query_vizier_catalog("II/335/galex_ais", TARGET_RA, TARGET_DEC, 6.0)
    if len(gx) > 0:
        add_band("GALEX_FUV", 0.1539, 3631.0, 2.65, safe_float(gx.iloc[0].get("FUVmag")), safe_float(gx.iloc[0].get("e_FUVmag")))
        add_band("GALEX_NUV", 0.2316, 3631.0, 2.75, safe_float(gx.iloc[0].get("NUVmag")), safe_float(gx.iloc[0].get("e_NUVmag")))

    rx = query_vizier_catalog("J/A+A/588/A103/cat2rxs", TARGET_RA, TARGET_DEC, 30.0)
    has_xray = len(rx) > 0

    sed = pd.DataFrame(bands).sort_values("lam_um").reset_index(drop=True)
    nu_hz = C_LIGHT / (sed["lam_um"].values * 1e-6)
    fnu_wm2hz = sed["fnu0_jy"].values * 1e-26 * np.power(10.0, -0.4 * sed["mag"].values)
    sed["lam_Fl_wm2"] = nu_hz * fnu_wm2hz
    sed["lam_Fl_err"] = sed["lam_Fl_wm2"] * (0.4 * np.log(10.0) * sed["mag_err"].values)
    return sed, has_xray


def planck_lam_fl(lam_um, teff, r_rsun, dist_pc, av, k_av):
    """Reddened blackbody lambda*F_lambda (W m^-2) at Earth."""
    lam_m = lam_um * 1e-6
    x = np.clip((H_PLANCK * C_LIGHT) / (lam_m * K_BOLTZ * teff), 1e-6, 700.0)
    b_lam = (2.0 * H_PLANCK * C_LIGHT ** 2 / (lam_m ** 5)) / np.expm1(x)
    ang_factor = (r_rsun * R_SUN_M / (dist_pc * PC_M)) ** 2
    ext_factor = np.power(10.0, -0.4 * av * k_av)
    return lam_m * np.pi * b_lam * ang_factor * ext_factor


def fit_sed_single_and_binary(sed: pd.DataFrame, dist_pc: float, av_prior: float):
    """Fit single-star and two-component blackbody SEDs."""
    lam = sed["lam_um"].values
    k_av = sed["k_av"].values
    y_obs = sed["lam_Fl_wm2"].values
    y_err = sed["lam_Fl_err"].values

    def res_single(p):
        teff, r_eff, av = p
        y_mod = planck_lam_fl(lam, teff, r_eff, dist_pc, av, k_av)
        r_phot = (y_obs - y_mod) / y_err
        if np.isfinite(av_prior):
            return np.append(r_phot, (av - av_prior) / 0.15)
        return r_phot

    p0 = [6500.0, 2.2, av_prior if np.isfinite(av_prior) else 0.35]
    lsq1 = least_squares(res_single, x0=p0, bounds=([4500.0, 0.5, 0.0], [9500.0, 8.0, 1.5]))
    teff1, r1_sed, av1 = lsq1.x
    dof1 = max(len(lam) - 3, 1)
    y_mod1 = planck_lam_fl(lam, teff1, r1_sed, dist_pc, av1, k_av)
    chi2_1 = float(np.sum(((y_obs - y_mod1) / y_err) ** 2))
    cov1 = (chi2_1 / dof1) * np.linalg.pinv(lsq1.jac.T @ lsq1.jac)
    err1 = np.sqrt(np.maximum(np.diag(cov1), 0.0))

    def res_binary(p):
        t1, r1, t2, r2, av = p
        y_mod = (
            planck_lam_fl(lam, t1, r1, dist_pc, av, k_av)
            + planck_lam_fl(lam, t2, r2, dist_pc, av, k_av)
        )
        r_phot = (y_obs - y_mod) / y_err
        if np.isfinite(av_prior):
            return np.append(r_phot, (av - av_prior) / 0.15)
        return r_phot

    lsq2 = least_squares(
        res_binary,
        x0=[6600.0, 1.9, 5500.0, 1.2, av1],
        bounds=([5500.0, 0.8, 3200.0, 0.2, 0.0], [8500.0, 5.0, 7500.0, 3.5, 1.5]),
    )
    y_mod2 = (
        planck_lam_fl(lam, lsq2.x[0], lsq2.x[1], dist_pc, lsq2.x[4], k_av)
        + planck_lam_fl(lam, lsq2.x[2], lsq2.x[3], dist_pc, lsq2.x[4], k_av)
    )
    chi2_2 = float(np.sum(((y_obs - y_mod2) / y_err) ** 2))

    return {
        "teff1": teff1, "teff1_err": err1[0],
        "r_eff": r1_sed, "r_eff_err": err1[1],
        "av": av1, "av_err": err1[2],
        "chi2_1": chi2_1, "dof1": dof1, "y_mod1": y_mod1,
        "bin_params": lsq2.x, "chi2_2": chi2_2, "y_mod2": y_mod2,
    }


def solve_kepler_and_epochs(p_orb, t_peri_gaia, ecc, omega_deg, t_ref_btjd):
    """
    Gaia DR3 nss_two_body_orbit reference epoch is BJD 2457389.0 (J2016.0 TCB).
    TESS BTJD = BJD - 2457000.0  =>  T_peri_BTJD = t_peri_gaia + 389.0.
    """
    t_peri_btjd = t_peri_gaia + 389.0
    n_cyc = round((t_ref_btjd - t_peri_btjd) / p_orb)
    t_peri_mid = t_peri_btjd + n_cyc * p_orb
    omega = np.radians(omega_deg)

    def nu_to_phase(nu_rad):
        e_anom = 2.0 * np.arctan2(
            np.sqrt(1.0 - ecc) * np.sin(nu_rad / 2.0),
            np.sqrt(1.0 + ecc) * np.cos(nu_rad / 2.0),
        )
        m_anom = (e_anom - ecc * np.sin(e_anom)) % (2.0 * np.pi)
        return float(m_anom / (2.0 * np.pi))

    return {
        "t_peri_btjd_orig": t_peri_btjd,
        "t_peri_btjd_mid": t_peri_mid,
        "n_cyc_from_gaia": n_cyc,
        "phi_rv_recede": nu_to_phase(0.0 - omega),
        "phi_rv_approach": nu_to_phase(np.pi - omega),
        "phi_conj_sup": nu_to_phase(0.5 * np.pi - omega),
        "phi_conj_inf": nu_to_phase(1.5 * np.pi - omega),
    }


def solve_m2_from_fm(fm, m1, inc_deg):
    """Solve m2^3 * sin^3(i) / (m1 + m2)^2 = fm."""
    sini = np.sin(np.radians(inc_deg))
    if sini <= 1e-4:
        return np.nan
    roots = np.roots([sini ** 3, -fm, -2.0 * fm * m1, -fm * m1 ** 2])
    real_pos = [np.real(r) for r in roots if abs(np.imag(r)) < 1e-8 and np.real(r) > 0]
    return float(min(real_pos)) if real_pos else np.nan


def doppler_beaming_alpha(teff, lam_um=0.7865):
    """Blackbody Doppler beaming factor x * exp(x) / (exp(x) - 1)."""
    x = (H_PLANCK * C_LIGHT) / (lam_um * 1e-6 * K_BOLTZ * teff)
    return float(x * np.exp(x) / np.expm1(x))


def main():
    report = []

    def log(msg=""):
        print(msg)
        report.append(msg)

    log("=" * 105)
    log("PHASE 4: COMPLETE PHYSICAL MODELING & PUBLICATION REPORT FOR TYC 4517-88-1 (TIC 397073349)")
    log(f"Periodogram Engine: {'Astropy Fast O(N log N)' if HAS_ASTROPY_LS else 'Chunked NumPy'} | Start Peak RAM: {peak_ram_mb():.1f} MB")
    log("=" * 105)

    # 1. Load TESS data + fetch Gaia DR3 & neighborhood
    tess, crowdsap_map = ensure_tess_data()
    g_row, df_nb = fetch_gaia_and_neighbors()

    log("\n[1] GAIA DR3 ASTROMETRIC, STELLAR & ORBITAL PARAMETERS")
    key_cols = [
        "source_id", "ra", "dec", "l", "b",
        "parallax", "parallax_error", "parallax_over_error",
        "pmra", "pmra_error", "pmdec", "pmdec_error",
        "ruwe", "ipd_frac_multi_peak", "non_single_star",
        "phot_g_mean_mag", "phot_bp_mean_mag", "phot_rp_mean_mag", "bp_rp", "phot_bp_rp_excess_factor",
        "radial_velocity", "radial_velocity_error", "rv_nb_transits", "rv_expected_sig_to_noise",
        "vbroad", "vbroad_error", "vbroad_nb_transits",
        "teff_gspphot", "teff_gspphot_lower", "teff_gspphot_upper",
        "logg_gspphot", "mh_gspphot", "ag_gspphot", "ebpminrp_gspphot",
        "radius_gspphot", "radius_flame", "radius_flame_lower", "radius_flame_upper",
        "lum_flame", "lum_flame_lower", "lum_flame_upper",
        "mass_flame", "mass_flame_lower", "mass_flame_upper", "age_flame",
        "teff_gspspec", "logg_gspspec", "mh_gspspec",
        "activityindex_espcs", "activityindex_espcs_uncertainty",
        "nss_solution_type", "period", "period_error",
        "t_periastron", "t_periastron_error",
        "eccentricity", "eccentricity_error",
        "center_of_mass_velocity", "center_of_mass_velocity_error",
        "semi_amplitude_primary", "semi_amplitude_primary_error",
        "arg_periastron", "arg_periastron_error",
        "significance", "goodness_of_fit", "efficiency",
    ]
    for c in key_cols:
        if c in g_row.index and pd.notna(g_row[c]):
            log(f"  {c:<30}: {g_row[c]}")

    plx = float(g_row["parallax"])
    plx_err = float(g_row["parallax_error"])
    dist_pc = 1000.0 / plx
    dist_err = dist_pc * (plx_err / plx)
    ag = float(g_row["ag_gspphot"]) if pd.notna(g_row["ag_gspphot"]) else 0.0
    av_gaia = ag / 0.836 if ag > 0 else np.nan
    mg_abs = float(g_row["phot_g_mean_mag"]) + 5.0 * np.log10(plx) - 10.0
    mg_dered = mg_abs - ag
    log(f"  {'dist_pc':<30}: {dist_pc:.2f} +/- {dist_err:.2f} pc")
    log(f"  {'M_G (raw / dereddened)':<30}: {mg_abs:.3f} / {mg_dered:.3f} mag (A_V_Gaia ~ {av_gaia:.3f})")

    # 2. 60-arcsec crowding audit
    rp0 = float(g_row["phot_rp_mean_mag"])
    df_nb["dRP"] = df_nb["phot_rp_mean_mag"] - rp0
    df_nb["flux_pct"] = 100.0 * np.power(10.0, -0.4 * df_nb["dRP"])
    nb_42 = df_nb[(df_nb["sep_arcsec"] > 0.5) & (df_nb["sep_arcsec"] <= 42.0)]
    cs_vals = [v for v in crowdsap_map.values() if np.isfinite(v)]
    log("\n[2] 60-ARCSEC CROWDING AUDIT")
    log(f"  Neighbors within 42\" (2 TESS pix): N={len(nb_42)}, total RP flux = {nb_42['flux_pct'].sum():.2f}% of target")
    if len(nb_42) > 0:
        brightest = nb_42.sort_values("flux_pct", ascending=False).iloc[0]
        log(f"  Brightest neighbor <= 42\": sep={brightest['sep_arcsec']:.1f}\", G={brightest['phot_g_mean_mag']:.2f}, flux_ratio={brightest['flux_pct']:.2f}%")
    if cs_vals:
        log(f"  TESS-SPOC CROWDSAP ({len(cs_vals)} sectors): mean={np.mean(cs_vals):.4f} (range {np.min(cs_vals):.4f} - {np.max(cs_vals):.4f})")

    # 3. Multi-band SED & Blackbody Fitting
    sed, has_xray = build_sed_table(g_row)
    sed.to_csv("tic397073349_sed_photometry.csv", index=False)
    sed_fit = fit_sed_single_and_binary(sed, dist_pc, av_gaia)

    log(f"\n[3] MULTI-WAVELENGTH ARCHIVAL SED ({len(sed)} BANDS) & X-RAY CHECK")
    log(sed[["band", "lam_um", "mag", "mag_err"]].to_string(index=False))
    log(f"  ROSAT 2RXS match within 30\" : {'YES' if has_xray else 'NONE (non-detection)'}")
    log(f"  Single-star BB fit : Teff = {sed_fit['teff1']:.0f} +/- {sed_fit['teff1_err']:.0f} K | "
        f"R_eff = {sed_fit['r_eff']:.3f} +/- {sed_fit['r_eff_err']:.3f} Rsun | "
        f"A_V = {sed_fit['av']:.3f} +/- {sed_fit['av_err']:.3f} | chi2/dof = {sed_fit['chi2_1']:.1f}/{sed_fit['dof1']}")
    bp = sed_fit["bin_params"]
    log(f"  Two-star BB fit    : T1={bp[0]:.0f} K, R1={bp[1]:.2f} Rsun + T2={bp[2]:.0f} K, R2={bp[3]:.2f} Rsun, A_V={bp[4]:.3f} | chi2 = {sed_fit['chi2_2']:.1f}")
    log(f"  Equal-luminosity twin radius (R_eff/sqrt(2)) = {sed_fit['r_eff']/np.sqrt(2.0):.3f} Rsun")

    # 4. Memory-Safe Orbital Period Refinement (scalar minimize_scalar on 2f+3f+4f)
    t_all = tess["btjd"].values.astype(np.float64)
    f_all = tess["flux_rel"].values.astype(np.float64)
    p_gaia = float(g_row["period"])
    p_gaia_err = float(g_row["period_error"])
    tp_gaia = float(g_row["t_periastron"])
    ecc = float(g_row["eccentricity"])
    ecc_err = float(g_row["eccentricity_error"])
    omega_deg = float(g_row["arg_periastron"])
    k1 = float(g_row["semi_amplitude_primary"])
    k1_err = float(g_row["semi_amplitude_primary_error"])
    fm = 1.036149e-7 * p_gaia * (k1 ** 3) * ((1.0 - ecc ** 2) ** 1.5)
    fm_err = fm * np.sqrt(
        (p_gaia_err / p_gaia) ** 2
        + (3.0 * k1_err / k1) ** 2
        + (3.0 * ecc * ecc_err / (1.0 - ecc ** 2)) ** 2
    )

    t_mid_all = float(np.mean(t_all))
    dt_mid = t_all - t_mid_all

    def rss_ell_period(p_val: float) -> float:
        f_o = 1.0 / p_val
        cols = [np.ones_like(dt_mid)]
        for k in (2, 3, 4):
            ang = 2.0 * np.pi * k * f_o * dt_mid
            cols.extend([np.cos(ang), np.sin(ang)])
        X = np.column_stack(cols)
        beta, _, _, _ = np.linalg.lstsq(X, f_all, rcond=None)
        r = f_all - X @ beta
        return float(np.dot(r, r))

    opt_p = minimize_scalar(rss_ell_period, bounds=(2.6270, 2.6286), method="bounded", options={"xatol": 1e-8})
    p_tess = float(opt_p.x)
    f_orb = 1.0 / p_tess
    ephem = solve_kepler_and_epochs(p_tess, tp_gaia, ecc, omega_deg, t_mid_all)
    t0_peri = ephem["t_peri_btjd_mid"]

    # Sector-by-sector harmonic decomposition anchored to exact Gaia periastron (BJD 2457389.0)
    sec_rows = []
    for s_id, grp in tess.groupby("sector"):
        ts = grp["btjd"].values.astype(np.float64)
        fs = grp["flux_rel"].values.astype(np.float64)
        dts = ts - t0_peri
        cols = [np.ones_like(dts)]
        for k in (1, 2, 3, 4):
            ang = 2.0 * np.pi * k * f_orb * dts
            cols.extend([np.cos(ang), np.sin(ang)])
        Xs = np.column_stack(cols)
        bs, _, _, _ = np.linalg.lstsq(Xs, fs, rcond=None)
        res_s = fs - Xs @ bs
        s2 = float(np.dot(res_s, res_s)) / max(len(ts) - Xs.shape[1], 1)
        cov_s = s2 * np.linalg.pinv(Xs.T @ Xs)
        err_s = np.sqrt(np.maximum(np.diag(cov_s), 0.0))

        a1, b1, a2, b2 = bs[1], bs[2], bs[3], bs[4]
        amp1 = float(np.hypot(a1, b1))
        amp2 = float(np.hypot(a2, b2))
        amp1_err = float(np.sqrt((a1 * err_s[1]) ** 2 + (b1 * err_s[2]) ** 2) / max(amp1, 1e-12))
        amp2_err = float(np.sqrt((a2 * err_s[3]) ** 2 + (b2 * err_s[4]) ** 2) / max(amp2, 1e-12))
        phi1 = float((np.arctan2(b1, a1) / (2.0 * np.pi)) % 1.0)
        phi2_max_orb = float((np.arctan2(b2, a2) / (4.0 * np.pi)) % 0.5)
        phi2_min_orb = float((phi2_max_orb + 0.25) % 0.5)

        sec_rows.append({
            "sector": int(s_id), "t_mid": float(np.mean(ts)), "npts": len(ts),
            "A1_ppm": 1e6 * amp1, "A1_err": 1e6 * amp1_err, "phi1": phi1,
            "a1_cos_ppm": 1e6 * a1, "b1_sin_ppm": 1e6 * b1,
            "A2_ppm": 1e6 * amp2, "A2_err": 1e6 * amp2_err,
            "phi2_max": phi2_max_orb, "phi2_min": phi2_min_orb,
            "rms_ppm": 1e6 * float(np.std(res_s)),
            "crowdsap": crowdsap_map.get(int(s_id), np.nan),
        })

    sec_df = pd.DataFrame(sec_rows).sort_values("t_mid").reset_index(drop=True)
    sec_times = sec_df["t_mid"].values
    sec_phi2_min = sec_df["phi2_min"].values
    ref_p = float(np.median(sec_phi2_min))
    sec_phi2_u = ref_p + ((sec_phi2_min - ref_p + 0.25) % 0.5) - 0.25

    t_c = sec_times - np.mean(sec_times)
    slope_phi, int_phi = np.polyfit(t_c, sec_phi2_u, 1)
    resid_phi = sec_phi2_u - (slope_phi * t_c + int_phi)
    se_slope = np.sqrt(np.sum(resid_phi ** 2) / max(len(t_c) - 2, 1)) / np.sqrt(np.sum(t_c ** 2))
    p_tess_refined = p_tess + slope_phi * (p_tess ** 2)
    p_tess_err_real = se_slope * (p_tess ** 2)

    # Global 4-harmonic fit on refined period
    f_orb_ref = 1.0 / p_tess_refined
    dt_all = t_all - t0_peri
    X_all = np.column_stack(
        [np.ones_like(dt_all)]
        + [fn(2.0 * np.pi * k * f_orb_ref * dt_all) for k in range(1, 5) for fn in (np.cos, np.sin)]
    )
    b_all, _, _, _ = np.linalg.lstsq(X_all, f_all, rcond=None)
    del X_all
    gc.collect()

    phi2_max_obs = float((np.arctan2(b_all[4], b_all[3]) / (4.0 * np.pi)) % 0.5)
    phi2_min_obs = float((phi2_max_obs + 0.25) % 0.5)
    a2_glob = 1e6 * float(np.hypot(b_all[3], b_all[4]))
    a2_sec_mean = float(sec_df["A2_ppm"].mean())
    a2_sec_std = float(sec_df["A2_ppm"].std(ddof=1))
    a2_sec_sem = a2_sec_std / np.sqrt(len(sec_df))

    pred_quad_mod = ephem["phi_rv_recede"] % 0.5
    pred_conj_mod = ephem["phi_conj_sup"] % 0.5
    prop_err_gaia = abs(ephem["n_cyc_from_gaia"]) * (p_gaia_err / p_gaia)
    dphi_align = ((phi2_max_obs - pred_quad_mod + 0.25) % 0.5) - 0.25

    log("\n[4] EXACT ORBITAL EPHEMERIS & GAIA-TESS PHASE ALIGNMENT (BJD 2457389.0 ZERO-POINT)")
    log(f"  Gaia DR3 SB1 Period               : {p_gaia:.6f} +/- {p_gaia_err:.6f} d")
    log(f"  TESS Refined Period (2f_orb)      : {p_tess_refined:.7f} +/- {p_tess_err_real:.7f} d (sector-timing 1-sigma)")
    log(f"  Period difference (TESS - Gaia)   : {(p_tess_refined - p_gaia)*86400.0:+.2f} s ({(p_tess_refined - p_gaia)/p_gaia_err:+.2f} sigma_Gaia)")
    log(f"  Orbits elapsed Gaia -> TESS       : {ephem['n_cyc_from_gaia']} cycles | Gaia phase prop err = +/-{prop_err_gaia:.3f} cyc")
    log(f"  Predicted RV Quadrature (mod 0.5) : {pred_quad_mod:.4f} | Observed Ellipsoidal Max (mod 0.5): {phi2_max_obs:.4f}")
    log(f"  Predicted Conjunction   (mod 0.5) : {pred_conj_mod:.4f} | Observed Ellipsoidal Min (mod 0.5): {phi2_min_obs:.4f}")
    log(f"  Phase Offset (Obs Max - Pred Quad): {dphi_align:+.4f} cyc ({dphi_align/max(prop_err_gaia,1e-4):+.2f} sigma of Gaia prop err)")
    log(f"  Ellipsoidal A2 (sector mean)      : {a2_sec_mean:.1f} +/- {a2_sec_sem:.1f} ppm (sector scatter = {a2_sec_std:.1f} ppm; global = {a2_glob:.1f} ppm)")

    # 5. Coherent Doppler Beaming Vector Test
    mean_c1 = float(sec_df["a1_cos_ppm"].mean())
    sem_c1 = float(sec_df["a1_cos_ppm"].std(ddof=1) / np.sqrt(len(sec_df)))
    mean_s1 = float(sec_df["b1_sin_ppm"].mean())
    sem_s1 = float(sec_df["b1_sin_ppm"].std(ddof=1) / np.sqrt(len(sec_df)))
    a1_vec = float(np.hypot(mean_c1, mean_s1))
    a1_vec_err = float(np.sqrt((mean_c1 * sem_c1) ** 2 + (mean_s1 * sem_s1) ** 2) / max(a1_vec, 1e-6))
    phi1_vec = float((np.arctan2(mean_s1, mean_c1) / (2.0 * np.pi)) % 1.0)
    ang_beam = 2.0 * np.pi * ephem["phi_rv_approach"]
    a1_beam_proj = float(mean_c1 * np.cos(ang_beam) + mean_s1 * np.sin(ang_beam))
    a1_beam_proj_err = float(np.hypot(sem_c1 * np.cos(ang_beam), sem_s1 * np.sin(ang_beam)))

    teff_use = float(g_row["teff_gspphot"]) if pd.notna(g_row["teff_gspphot"]) else sed_fit["teff1"]
    alpha_beam_bb = doppler_beaming_alpha(teff_use, lam_um=0.7865)
    alpha_beam_tess = 2.65
    pred_beam_dark_ppm = 1e6 * alpha_beam_tess * (k1 * 1000.0 / C_LIGHT)
    a1_sorted = np.sort(sec_df["A1_ppm"].values)

    log("\n[5] DOPPLER BEAMING & STARSPOT DIAGNOSTICS")
    log(f"  Binary mass function f(M)            : {fm:.4f} +/- {fm_err:.4f} Msun")
    log(f"  Predicted undiluted beaming (L2=0)   : {pred_beam_dark_ppm:.0f} ppm (B_TESS={alpha_beam_tess:.2f}, BB={alpha_beam_bb:.2f})")
    log(f"  Observed sector-mean coherent A1_vec : {a1_vec:.1f} +/- {a1_vec_err:.1f} ppm (at phi={phi1_vec:.3f} vs RV approach {ephem['phi_rv_approach']:.3f})")
    log(f"  Projected beaming component along RV : {a1_beam_proj:+.1f} +/- {a1_beam_proj_err:.1f} ppm")
    log(f"  Low-spot floor (4 quietest sectors)  : A1 = {a1_sorted[0]:.1f} - {a1_sorted[min(3, len(a1_sorted)-1)]:.1f} ppm")

    # 6. Companion Mass & Inclination Physical Modeling
    m1_est = float(g_row["mass_flame"]) if pd.notna(g_row["mass_flame"]) else 1.45
    r1_flame = float(g_row["radius_flame"]) if pd.notna(g_row["radius_flame"]) else sed_fit["r_eff"]
    u_ld, tau_gd = 0.35, 0.35
    c_ell = 0.15 * (15.0 + u_ld) / (3.0 - u_ld) * (1.0 + tau_gd)

    inc_grid = np.linspace(15.0, 90.0, 301)
    m2_grid = np.array([solve_m2_from_fm(fm, m1_est, inc) for inc in inc_grid])
    q_grid = m2_grid / m1_est
    a_grid = 4.206 * np.power(m1_est + m2_grid, 1.0 / 3.0) * np.power(p_tess_refined, 2.0 / 3.0)

    a2_dark_ppm = 1e6 * c_ell * q_grid * np.power(r1_flame / a_grid, 3.0) * (np.sin(np.radians(inc_grid)) ** 2)

    r2_ms = np.power(m2_grid, 0.85)
    t2_ms = 5772.0 * np.power(m2_grid, 0.55)
    r1_lum = np.sqrt(np.maximum(r1_flame ** 2 - (r2_ms ** 2) * np.power(t2_ms / teff_use, 2.5), 0.9 ** 2))
    l1_frac = (r1_lum ** 2) / (r1_lum ** 2 + (r2_ms ** 2) * np.power(t2_ms / teff_use, 2.5))
    l2_frac = 1.0 - l1_frac
    a2_1 = c_ell * q_grid * np.power(r1_lum / a_grid, 3.0) * (np.sin(np.radians(inc_grid)) ** 2)
    a2_2 = c_ell * (1.0 / q_grid) * np.power(r2_ms / a_grid, 3.0) * (np.sin(np.radians(inc_grid)) ** 2)
    a2_lum_ppm = 1e6 * (l1_frac * a2_1 + l2_frac * a2_2)
    b2_arr = np.array([doppler_beaming_alpha(t2, 0.7865) * 0.88 for t2 in t2_ms])
    beam_lum_ppm = 1e6 * (k1 * 1000.0 / C_LIGHT) * np.abs(l1_frac * alpha_beam_tess - l2_frac * b2_arr / q_grid)

    inc_no_ecl_lum = np.degrees(np.arccos(np.clip((r1_lum + r2_ms) / a_grid, 0.0, 0.99)))
    idx_dark = int(np.argmin(np.abs(a2_dark_ppm - a2_sec_mean)))
    idx_lum = int(np.argmin(np.abs(a2_lum_ppm - a2_sec_mean)))

    vbroad = safe_float(g_row["vbroad"])
    vbroad_err = safe_float(g_row["vbroad_error"])
    v_sync_dark = 50.579 * r1_flame / p_tess_refined
    v_sync_lum = 50.579 * r1_lum[idx_lum] / p_tess_refined

    log("\n[6] PHYSICAL SOLUTIONS FOR COMPANION MASS M2 & INCLINATION i")
    log(f"  Adopted primary M1 = {m1_est:.2f} Msun, R_eff = {r1_flame:.2f} Rsun (SED R_eff = {sed_fit['r_eff']:.2f} Rsun)")
    log(f"  Absolute minimum companion mass (i = 90 deg) : M2_min = {m2_grid[-1]:.3f} Msun")
    log("  --- SCENARIO A: DARK COMPACT COMPANION (L2 = 0) ---")
    log(f"    Matches A2={a2_sec_mean:.0f} ppm at i = {inc_grid[idx_dark]:.1f} deg => M2 = {m2_grid[idx_dark]:.2f} Msun (q = {q_grid[idx_dark]:.2f}, a = {a_grid[idx_dark]:.1f} Rsun)")
    log(f"    Predicted Doppler beaming = {pred_beam_dark_ppm:.0f} ppm")
    log("  --- SCENARIO B: LUMINOUS MS SECONDARY (Flux Dilution + Beaming Cancellation) ---")
    log(f"    Matches A2={a2_sec_mean:.0f} ppm at i = {inc_grid[idx_lum]:.1f} deg => M2 = {m2_grid[idx_lum]:.2f} Msun (q = {q_grid[idx_lum]:.2f}, R1={r1_lum[idx_lum]:.2f}, R2={r2_ms[idx_lum]:.2f} Rsun, L2/Ltot={l2_frac[idx_lum]:.2f})")
    log(f"    Predicted net Doppler beaming = {beam_lum_ppm[idx_lum]:.0f} ppm")
    log(f"    No-eclipse ceiling at this M2 : i < {inc_no_ecl_lum[idx_lum]:.1f} deg")
    if np.isfinite(vbroad):
        log("  --- GAIA DR3 VBROAD ROTATIONAL INCLINATION CHECK ---")
        log(f"    Gaia vbroad = {vbroad:.1f} +/- {vbroad_err:.1f} km/s | v_sync(R_eff) = {v_sync_dark:.1f} km/s -> sin(i) ~ {vbroad / v_sync_dark:.2f}")
        log(f"    v_sync(R1_lum={r1_lum[idx_lum]:.2f}) = {v_sync_lum:.1f} km/s -> sin(i) ~ {vbroad / v_sync_lum:.2f}")

    # =========================================================================
    # PUBLICATION FIGURES (MEMORY-SAFE, 300-DPI)
    # =========================================================================
    plt.rcParams.update({"font.size": 10, "axes.linewidth": 1.0})

    # --- FIG 1: 8-Sector Phase-Folded Light Curve + Multi-Year Sector Stability ---
    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(8.5, 9.2), gridspec_kw={"height_ratios": [2.2, 1.0, 1.0]}, constrained_layout=True
    )
    phase_obs = ((t_all - t0_peri) / p_tess_refined) % 1.0
    edges = np.linspace(0.0, 1.0, 101)
    ctrs = 0.5 * (edges[:-1] + edges[1:])
    b_med = np.array([np.median(f_all[(phase_obs >= edges[i]) & (phase_obs < edges[i + 1])]) for i in range(100)])
    b_err = np.array([
        1.4826 * np.median(np.abs(f_all[(phase_obs >= edges[i]) & (phase_obs < edges[i + 1])] - b_med[i]))
        / np.sqrt(max(np.sum((phase_obs >= edges[i]) & (phase_obs < edges[i + 1])), 1))
        for i in range(100)
    ])
    p_fine = np.linspace(0.0, 1.0, 1000)
    mod_fine = np.full_like(p_fine, b_all[0])
    for k in range(1, 5):
        mod_fine += b_all[2 * k - 1] * np.cos(2 * np.pi * k * p_fine) + b_all[2 * k] * np.sin(2 * np.pi * k * p_fine)

    ax1.errorbar(ctrs, 1e6 * b_med, yerr=1e6 * b_err, fmt="o", ms=3.5, color="#1f4e79", ecolor="#9dbcd4", capsize=1.5, label="8-sector binned TESS photometry")
    ax1.plot(p_fine, 1e6 * mod_fine, color="#c0392b", lw=2.0, label="4-harmonic orbital model")
    ax1.axvline(ephem["phi_rv_approach"], color="#1e8449", ls="--", lw=1.2, label=f"Gaia RV max approach ($\\phi={ephem['phi_rv_approach']:.2f}$)")
    ax1.axvline(ephem["phi_rv_recede"], color="#d35400", ls=":", lw=1.3, label=f"Gaia RV max recede ($\\phi={ephem['phi_rv_recede']:.2f}$)")
    ax1.set_xlim(0, 1)
    ax1.set_xlabel("Orbital Phase $\\phi$ (anchored to Gaia DR3 $T_p$, BJD 2457389.0)")
    ax1.set_ylabel(r"$\Delta F / F$ (ppm)")
    ax1.set_title(f"TYC 4517-88-1 (TIC 397073349): 8-Sector TESS Phase Fold ($P = {p_tess_refined:.6f}$ d)", fontweight="bold")
    ax1.legend(fontsize=8.5, ncol=2, loc="lower center")
    ax1.grid(alpha=0.25)

    ax2.errorbar(sec_df["t_mid"], sec_df["A2_ppm"], yerr=sec_df["A2_err"], fmt="s-", color="#c0392b", label=r"$A_2$ ($2f_{\rm orb}$, ellipsoidal)")
    ax2.errorbar(sec_df["t_mid"], sec_df["A1_ppm"], yerr=sec_df["A1_err"], fmt="o-", color="#117a65", label=r"$A_1$ ($1f_{\rm orb}$, spots + beaming)")
    ax2.axhline(pred_beam_dark_ppm, color="#555555", ls="--", lw=1.0, label=f"Undiluted beaming ($L_2=0$, {pred_beam_dark_ppm:.0f} ppm)")
    ax2.set_ylabel("Semi-Amplitude (ppm)")
    ax2.legend(fontsize=8.2, ncol=3, loc="upper right")
    ax2.grid(alpha=0.25)

    ax3.plot(sec_times, sec_phi2_u, "s-", color="#c0392b", label=r"$\phi_{\rm min}(2f_{\rm orb})$ (tidal lock)")
    ax3.plot(sec_df["t_mid"], sec_df["phi1"], "o-", color="#117a65", label=r"$\phi_{\rm max}(1f_{\rm orb})$ (migrating spots)")
    ax3.set_xlabel("Sector Midpoint Epoch (BTJD)")
    ax3.set_ylabel("Phase (cycles)")
    ax3.legend(fontsize=8.5, ncol=2, loc="upper right")
    ax3.grid(alpha=0.25)

    fig.savefig("fig1_tess_phase_and_sectors.png", dpi=300)
    plt.close(fig)
    gc.collect()

    # --- FIG 2: Residual Spectrum & Hut Pseudo-Synchronization (O(N log N) Fast LS) ---
    X_no1 = np.column_stack(
        [np.ones_like(dt_mid)]
        + [fn(2 * np.pi * k * f_orb_ref * dt_mid) for k in (2, 3, 4) for fn in (np.cos, np.sin)]
    )
    b_no1, _, _, _ = np.linalg.lstsq(X_no1, f_all, rcond=None)
    res_no2 = f_all - X_no1 @ b_no1
    del X_no1
    gc.collect()

    freqs_lo = np.linspace(0.15, 1.35, 25000)
    amp_lo = fast_lombscargle_amp_ppm(dt_mid, res_no2, freqs_lo)

    num_h = 1 + 7.5 * ecc ** 2 + (45 / 8) * ecc ** 4 + (5 / 16) * ecc ** 6
    den_h = (1 + 3 * ecc ** 2 + (3 / 8) * ecc ** 4) * ((1 - ecc ** 2) ** 1.5)
    f_ps = f_orb_ref * (num_h / den_h)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.3), constrained_layout=True)
    ax1.plot(freqs_lo, amp_lo, color="#2c3e50", lw=0.9)
    ax1.axvline(f_orb_ref, color="#c0392b", ls="--", lw=1.2, label=f"$f_{{\\rm orb}} = {f_orb_ref:.4f}\\,\\mathrm{{d}}^{{-1}}$")
    ax1.axvline(f_ps, color="#1e8449", ls="-.", lw=1.3, label=f"Hut (1981) $f_{{\\rm ps}} = {f_ps:.4f}\\,\\mathrm{{d}}^{{-1}}$")
    ax1.axvline(2 * f_orb_ref, color="#c0392b", ls=":", lw=1.0, alpha=0.7, label=r"$2f_{\rm orb}$ (subtracted)")
    ax1.set_xlim(0.20, 1.00)
    ax1.set_xlabel(r"Frequency ($\mathrm{d}^{-1}$)")
    ax1.set_ylabel("Amplitude (ppm)")
    ax1.set_title("Post-$2f_{\\rm orb}$ Residual Amplitude Spectrum", fontweight="bold")
    ax1.legend(fontsize=8.5)
    ax1.grid(alpha=0.25)

    f_scan = np.linspace(0.28, 0.48, 4000)
    p_rot_secs, amp_rot_secs = [], []
    for _, grp in tess.groupby("sector"):
        ts = grp["btjd"].values.astype(np.float64)
        fs = grp["flux_rel"].values.astype(np.float64)
        dts = ts - np.mean(ts)
        Xs = np.column_stack([np.ones_like(dts), np.cos(4 * np.pi * f_orb_ref * dts), np.sin(4 * np.pi * f_orb_ref * dts)])
        bs, _, _, _ = np.linalg.lstsq(Xs, fs, rcond=None)
        rs = fs - Xs @ bs
        amps = fast_lombscargle_amp_ppm(dts, rs, f_scan)
        p_rot_secs.append(1.0 / f_scan[int(np.argmax(amps))])
        amp_rot_secs.append(float(np.max(amps)))

    sc = ax2.scatter(sec_times, p_rot_secs, c=amp_rot_secs, cmap="viridis", s=75, edgecolor="k", zorder=4)
    ax2.axhline(p_tess_refined, color="#c0392b", ls="--", lw=1.2, label=f"$P_{{\\rm orb}} = {p_tess_refined:.3f}$ d")
    ax2.axhline(1.0 / f_ps, color="#1e8449", ls="-.", lw=1.3, label=f"$P_{{\\rm ps}}$ (Hut 1981) $= {1/f_ps:.3f}$ d")
    cbar = plt.colorbar(sc, ax=ax2)
    cbar.set_label("Spot Semi-Amplitude (ppm)")
    ax2.set_xlabel("Sector Midpoint Epoch (BTJD)")
    ax2.set_ylabel("Dominant Spot Period $P_{\\rm rot}$ (d)")
    ax2.set_title("Migrating Near-Synchronous Spot Period Across 4.6 yr", fontweight="bold")
    ax2.legend(fontsize=8.5, loc="upper left")
    ax2.grid(alpha=0.25)

    fig.savefig("fig2_frequency_and_spots.png", dpi=300)
    plt.close(fig)
    gc.collect()

    # --- FIG 3: Multi-Band Archival SED ---
    fig, ax = plt.subplots(figsize=(7.6, 5.0), constrained_layout=True)
    lam_grid = np.logspace(np.log10(0.15), np.log10(15.0), 400)
    k_smooth = np.interp(lam_grid, sed["lam_um"].values, sed["k_av"].values)
    y_line1 = planck_lam_fl(lam_grid, sed_fit["teff1"], sed_fit["r_eff"], dist_pc, sed_fit["av"], k_smooth)
    y_line2 = (
        planck_lam_fl(lam_grid, bp[0], bp[1], dist_pc, bp[4], k_smooth)
        + planck_lam_fl(lam_grid, bp[2], bp[3], dist_pc, bp[4], k_smooth)
    )
    ax.loglog(lam_grid, y_line1, color="#c0392b", lw=1.8, label=f"Single BB ($T={sed_fit['teff1']:.0f}$ K, $R_{{\\rm eff}}={sed_fit['r_eff']:.2f}\\,R_\\odot$, $A_V={sed_fit['av']:.2f}$)")
    ax.loglog(lam_grid, y_line2, color="#117a65", ls="--", lw=1.6, label=f"Two-star BB ($T_1={bp[0]:.0f}$ K + $T_2={bp[2]:.0f}$ K)")
    ax.errorbar(sed["lam_um"], sed["lam_Fl_wm2"], yerr=sed["lam_Fl_err"], fmt="o", ms=5.5, color="#1f4e79", ecolor="#5d6d7e", capsize=2.5, label="Archival photometry (Gaia, Tycho, APASS, 2MASS, WISE)")
    ax.set_xlabel(r"Wavelength $\lambda$ ($\mu\mathrm{m}$)")
    ax.set_ylabel(r"$\lambda F_\lambda$ ($\mathrm{W\,m^{-2}}$)")
    ax.set_title("TYC 4517-88-1: Multi-Wavelength Spectral Energy Distribution", fontweight="bold")
    ax.set_xlim(0.25, 14.0)
    ax.legend(fontsize=8.2, loc="lower left")
    ax.grid(True, which="both", alpha=0.25)

    fig.savefig("fig3_sed_and_hr_position.png", dpi=300)
    plt.close(fig)
    gc.collect()

    # --- FIG 4: Companion Mass vs Inclination + Ellipsoidal & Beaming Constraints ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True)

    ax1.plot(inc_grid, a2_dark_ppm, color="#8e44ad", lw=2.0, label="Dark compact companion ($L_2 = 0$)")
    ax1.plot(inc_grid, a2_lum_ppm, color="#1f618d", lw=2.0, label="Luminous MS companion (diluted)")
    ax1.axhspan(a2_sec_mean - a2_sec_std, a2_sec_mean + a2_sec_std, color="#e74c3c", alpha=0.20, label=f"Observed $A_2 = {a2_sec_mean:.0f} \\pm {a2_sec_std:.0f}$ ppm")
    ax1.axhline(a2_sec_mean, color="#c0392b", ls="--", lw=1.2)
    ax1.axvline(inc_no_ecl_lum[idx_lum], color="#7f8c8d", ls=":", lw=1.2, label=f"No-eclipse ceiling ($i < {inc_no_ecl_lum[idx_lum]:.0f}^\\circ$)")
    ax1.set_xlim(15, 90)
    ax1.set_ylim(0, max(6500, np.nanmax(a2_dark_ppm) * 1.05))
    ax1.set_xlabel("Orbital Inclination $i$ (deg)")
    ax1.set_ylabel("Predicted Ellipsoidal $A_2$ (ppm)")
    ax1.set_title("Ellipsoidal Amplitude Constraint on Inclination", fontweight="bold")
    ax1.legend(fontsize=8.0, loc="upper left")
    ax1.grid(alpha=0.25)

    ax2.plot(inc_grid, m2_grid, color="#2c3e50", lw=2.2, label=f"$M_2(i)$ for $M_1={m1_est:.2f}\\,M_\\odot$, $f(M)={fm:.3f}\\,M_\\odot$")
    ax2.axhline(1.40, color="#d35400", ls="--", lw=1.1, label="Chandrasekhar limit ($1.40\\,M_\\odot$)")
    ax2.scatter([inc_grid[idx_dark]], [m2_grid[idx_dark]], color="#8e44ad", s=80, zorder=5, label=f"Dark solution: $i={inc_grid[idx_dark]:.0f}^\\circ, M_2={m2_grid[idx_dark]:.2f}\\,M_\\odot$")
    ax2.scatter([inc_grid[idx_lum]], [m2_grid[idx_lum]], color="#1f618d", s=80, marker="D", zorder=5, label=f"Luminous MS: $i={inc_grid[idx_lum]:.0f}^\\circ, M_2={m2_grid[idx_lum]:.2f}\\,M_\\odot$")
    ax2.set_xlim(15, 90)
    ax2.set_ylim(0.8, 5.5)
    ax2.set_xlabel("Orbital Inclination $i$ (deg)")
    ax2.set_ylabel(r"Companion Mass $M_2$ ($M_\odot$)")
    ax2.set_title("Companion Mass Constraints ($M_2$ vs. $i$)", fontweight="bold")
    ax2.legend(fontsize=8.0, loc="upper right")
    ax2.grid(alpha=0.25)

    fig.savefig("fig4_companion_constraints.png", dpi=300)
    plt.close(fig)
    gc.collect()

    log(f"\n[DONE] Peak RAM used: {peak_ram_mb():.1f} MB")
    log("[SAVED] phase4_paper_numbers.txt, tic397073349_sed_photometry.csv, and fig1..fig4 (*.png)")

    with open("phase4_paper_numbers.txt", "w", encoding="utf-8") as fh:
        fh.write("\n".join(report) + "\n")


if __name__ == "__main__":
    main()

PHASE 4: COMPLETE PHYSICAL MODELING & PUBLICATION REPORT FOR TYC 4517-88-1 (TIC 397073349)
Periodogram Engine: Astropy Fast O(N log N) | Start Peak RAM: 355.9 MB

[1] GAIA DR3 ASTROMETRIC, STELLAR & ORBITAL PARAMETERS
  source_id                     : 555926586677752960
  ra                            : 54.613980049872005
  dec                           : 79.34350268825605
  l                             : 130.4148698285385
  b                             : 18.98643784685197
  parallax                      : 1.687112001867416
  parallax_error                : 0.011869555
  parallax_over_error           : 142.13776
  pmra                          : 12.262580160900765
  pmra_error                    : 0.012507558
  pmdec                         : -12.917226184700525
  pmdec_error                   : 0.012974393
  ruwe                          : 0.79920894
  ipd_frac_multi_peak           : 0
  non_single_star               : 2
  phot_g_mean_mag               : 11.531674
  phot_bp_mean_mag

In [6]:
#!/usr/bin/env python3
"""
Phase 4B: Publication-Grade Figure Polish (Vector PDF + 300-DPI PNG)
Fixes the two visual issues caught during Phase 4 audit:
  - Fig 3: Fits photospheric BB strictly on 16 optical-to-MIR bands (0.42-11.6 um),
           displays GALEX FUV/NUV (0.154, 0.232 um) alongside a 20,000 K hot-WD
           overlay to visually rule out a hot WD, and adds a flux-ratio residual panel.
  - Fig 4: Removes the i=53 deg clamping kink by plotting the self-consistent F+F/G
           hidden-SB2 solution (M1=1.32 Msun, R_eff=2.06 Rsun split by flux) over its
           physically allowed luminosity domain, and adds a third panel showing the
           >5-sigma Doppler beaming veto against the 5 Msun dark-companion impostor.
  - Fig 1 & Fig 2: Clean legend placement (zero overlap with data curves) + PDF export.
"""

import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from astropy.timeseries import LombScargle as AstropyLS

warnings.filterwarnings("ignore")

# Verified Phase 4 Constants for TYC 4517-88-1 (TIC 397073349 / Gaia DR3 555926586677752960)
DIST_PC = 592.73
P_TESS = 2.6278035
P_GAIA = 2.6279587
TP_GAIA = -1.14072245
ECC = 0.0766532
OMEGA_DEG = 256.0386
K1_KMS = 93.9548
FM_MSUN = 0.2239
TEFF_GAIA = 6536.5
R_FLAME = 2.0559
M_FLAME = 1.4909
AV_GAIA = 0.197
A2_MEAN_PPM = 1821.3
A2_STD_PPM = 201.6
A1_BEAM_PROJ_PPM = 101.9
A1_BEAM_ERR_PPM = 133.1
A1_QUIET_FLOOR_PPM = 165.0

H_PLANCK = 6.62607015e-34
C_LIGHT = 2.99792458e8
K_BOLTZ = 1.380649e-23
R_SUN_M = 6.957e8
PC_M = 3.085677581e16


def planck_lam_fl(lam_um, teff, r_rsun, dist_pc, av, k_av):
    lam_m = lam_um * 1e-6
    x = np.clip((H_PLANCK * C_LIGHT) / (lam_m * K_BOLTZ * teff), 1e-6, 700.0)
    b_lam = (2.0 * H_PLANCK * C_LIGHT ** 2 / (lam_m ** 5)) / np.expm1(x)
    return lam_m * np.pi * b_lam * (r_rsun * R_SUN_M / (dist_pc * PC_M)) ** 2 * np.power(10.0, -0.4 * av * k_av)


def solve_m2(fm, m1, inc_deg):
    sini = np.sin(np.radians(inc_deg))
    roots = np.roots([sini ** 3, -fm, -2.0 * fm * m1, -fm * m1 ** 2])
    pos = [np.real(r) for r in roots if abs(np.imag(r)) < 1e-8 and np.real(r) > 0]
    return float(min(pos)) if pos else np.nan


def solve_epochs():
    t_peri_btjd = TP_GAIA + 389.0
    tess = pd.read_csv("tic397073349_all_tess.csv")
    t_mid = float(tess["btjd"].mean())
    n_cyc = round((t_mid - t_peri_btjd) / P_TESS)
    t0_peri = t_peri_btjd + n_cyc * P_TESS
    omega = np.radians(OMEGA_DEG)

    def nu_to_phase(nu_rad):
        ea = 2.0 * np.arctan2(np.sqrt(1 - ECC) * np.sin(nu_rad / 2), np.sqrt(1 + ECC) * np.cos(nu_rad / 2))
        return float(((ea - ECC * np.sin(ea)) % (2 * np.pi)) / (2 * np.pi))

    return tess, t0_peri, nu_to_phase(0.0 - omega), nu_to_phase(np.pi - omega)


def main():
    plt.rcParams.update({"font.size": 10, "axes.linewidth": 1.0, "font.family": "serif"})

    tess, t0_peri, phi_recede, phi_approach = solve_epochs()
    sed = pd.read_csv("tic397073349_sed_photometry.csv")

    t_all = tess["btjd"].values.astype(np.float64)
    f_all = tess["flux_rel"].values.astype(np.float64)
    f_orb = 1.0 / P_TESS
    dt_all = t_all - t0_peri

    # Global 4-harmonic fit & per-sector fits
    X_all = np.column_stack(
        [np.ones_like(dt_all)]
        + [fn(2 * np.pi * k * f_orb * dt_all) for k in range(1, 5) for fn in (np.cos, np.sin)]
    )
    b_all, _, _, _ = np.linalg.lstsq(X_all, f_all, rcond=None)
    del X_all

    sec_rows = []
    for s_id, grp in tess.groupby("sector"):
        ts = grp["btjd"].values.astype(np.float64)
        fs = grp["flux_rel"].values.astype(np.float64)
        dts = ts - t0_peri
        Xs = np.column_stack(
            [np.ones_like(dts)]
            + [fn(2 * np.pi * k * f_orb * dts) for k in range(1, 5) for fn in (np.cos, np.sin)]
        )
        bs, _, _, _ = np.linalg.lstsq(Xs, fs, rcond=None)
        res_s = fs - Xs @ bs
        s2 = float(np.dot(res_s, res_s)) / max(len(ts) - Xs.shape[1], 1)
        err_s = np.sqrt(np.maximum(np.diag(s2 * np.linalg.pinv(Xs.T @ Xs)), 0.0))
        a1, b1, a2, b2 = bs[1], bs[2], bs[3], bs[4]
        amp1, amp2 = float(np.hypot(a1, b1)), float(np.hypot(a2, b2))
        e1 = float(np.sqrt((a1 * err_s[1]) ** 2 + (b1 * err_s[2]) ** 2) / max(amp1, 1e-12))
        e2 = float(np.sqrt((a2 * err_s[3]) ** 2 + (b2 * err_s[4]) ** 2) / max(amp2, 1e-12))
        phi1 = float((np.arctan2(b1, a1) / (2 * np.pi)) % 1.0)
        phi2_min = float(((np.arctan2(b2, a2) / (4 * np.pi)) % 0.5 + 0.25) % 0.5)
        sec_rows.append({
            "sector": int(s_id), "t_mid": float(np.mean(ts)),
            "A1": 1e6 * amp1, "A1_err": 1e6 * e1, "phi1": phi1,
            "A2": 1e6 * amp2, "A2_err": 1e6 * e2, "phi2_min": phi2_min,
        })
    sdf = pd.DataFrame(sec_rows).sort_values("t_mid").reset_index(drop=True)
    ref_p = float(np.median(sdf["phi2_min"]))
    phi2_u = ref_p + ((sdf["phi2_min"].values - ref_p + 0.25) % 0.5) - 0.25

    # =========================================================================
    # FIG 1: 8-Sector Phase-Folded Light Curve + Multi-Year Stability (Clean Legends)
    # =========================================================================
    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(8.2, 9.0), gridspec_kw={"height_ratios": [2.2, 1.05, 1.0]}, constrained_layout=True
    )
    phase_obs = ((t_all - t0_peri) / P_TESS) % 1.0
    edges = np.linspace(0, 1, 101)
    ctrs = 0.5 * (edges[:-1] + edges[1:])
    b_med = np.array([np.median(f_all[(phase_obs >= edges[i]) & (phase_obs < edges[i + 1])]) for i in range(100)])
    b_err = np.array([
        1.4826 * np.median(np.abs(f_all[(phase_obs >= edges[i]) & (phase_obs < edges[i + 1])] - b_med[i]))
        / np.sqrt(max(np.sum((phase_obs >= edges[i]) & (phase_obs < edges[i + 1])), 1))
        for i in range(100)
    ])
    pf = np.linspace(0, 1, 1000)
    mf = np.full_like(pf, b_all[0])
    for k in range(1, 5):
        mf += b_all[2 * k - 1] * np.cos(2 * np.pi * k * pf) + b_all[2 * k] * np.sin(2 * np.pi * k * pf)

    ax1.errorbar(ctrs, 1e6 * b_med, yerr=1e6 * b_err, fmt="o", ms=3.2, color="#1f4e79", ecolor="#9dbcd4", capsize=1.5, label="8-sector binned TESS")
    ax1.plot(pf, 1e6 * mf, color="#c0392b", lw=2.0, label="4-harmonic orbital fit")
    ax1.axvline(phi_approach, color="#1e8449", ls="--", lw=1.2, label=f"Gaia RV approach ($\\phi={phi_approach:.2f}$)")
    ax1.axvline(phi_recede, color="#d35400", ls=":", lw=1.3, label=f"Gaia RV recede ($\\phi={phi_recede:.2f}$)")
    ax1.set_xlim(0, 1)
    ax1.set_ylim(-2550, 2950)
    ax1.set_xlabel("Orbital Phase $\\phi$ (anchored to Gaia DR3 $T_p$, BJD 2457389.0)")
    ax1.set_ylabel(r"Relative Flux $\Delta F / F$ (ppm)")
    ax1.set_title(f"TYC 4517-88-1 (TIC 397073349): 8-Sector TESS Phase Fold ($P = {P_TESS:.6f}$ d)", fontweight="bold")
    ax1.legend(fontsize=8.3, ncol=2, loc="upper left", framealpha=0.92)
    ax1.grid(alpha=0.25)

    ax2.errorbar(sdf["t_mid"], sdf["A2"], yerr=sdf["A2_err"], fmt="s-", color="#c0392b", label=r"$A_2$ ($2f_{\rm orb}$, ellipsoidal)")
    ax2.errorbar(sdf["t_mid"], sdf["A1"], yerr=sdf["A1_err"], fmt="o-", color="#117a65", label=r"$A_1$ ($1f_{\rm orb}$, spots + beaming)")
    ax2.axhline(831.0, color="#444444", ls="--", lw=1.1, label="Undiluted beaming ($L_2=0$, 831 ppm)")
    ax2.set_ylim(0, 2550)
    ax2.set_ylabel("Semi-Amplitude (ppm)")
    ax2.legend(fontsize=8.0, ncol=3, loc="upper center", framealpha=0.92)
    ax2.grid(alpha=0.25)

    ax3.plot(sdf["t_mid"], phi2_u, "s-", color="#c0392b", label=r"$\phi_{\rm min}(2f_{\rm orb})$ (tidal lock)")
    ax3.plot(sdf["t_mid"], sdf["phi1"], "o-", color="#117a65", label=r"$\phi_{\rm max}(1f_{\rm orb})$ (migrating spots)")
    ax3.set_ylim(-0.05, 1.18)
    ax3.set_xlabel("Sector Midpoint Epoch (BTJD)")
    ax3.set_ylabel("Phase (cycles)")
    ax3.legend(fontsize=8.3, ncol=2, loc="upper left", framealpha=0.92)
    ax3.grid(alpha=0.25)

    for ext in ("png", "pdf"):
        fig.savefig(f"fig1_tess_phase_and_sectors.{ext}", dpi=300)
    plt.close(fig)
    gc.collect()

    # =========================================================================
    # FIG 2: Residual Amplitude Spectrum + Hut (1981) Pseudo-Synchronization
    # =========================================================================
    dt_mid = t_all - float(np.mean(t_all))
    X_no1 = np.column_stack(
        [np.ones_like(dt_mid)]
        + [fn(2 * np.pi * k * f_orb * dt_mid) for k in (2, 3, 4) for fn in (np.cos, np.sin)]
    )
    b_no1, _, _, _ = np.linalg.lstsq(X_no1, f_all, rcond=None)
    res_no2 = f_all - X_no1 @ b_no1
    del X_no1

    freqs_lo = np.linspace(0.15, 1.35, 25000)
    y0 = res_no2 - np.mean(res_no2)
    pgram = AstropyLS(dt_mid, y0, center_data=True, fit_mean=True).power(freqs_lo, method="fast", assume_regular_frequency=True)
    amp_lo = 1e6 * np.sqrt(2.0 * np.clip(pgram, 0, 1) * np.var(y0))

    num_h = 1 + 7.5 * ECC ** 2 + (45 / 8) * ECC ** 4 + (5 / 16) * ECC ** 6
    den_h = (1 + 3 * ECC ** 2 + (3 / 8) * ECC ** 4) * ((1 - ECC ** 2) ** 1.5)
    f_ps = f_orb * (num_h / den_h)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.2, 4.2), constrained_layout=True)
    ax1.plot(freqs_lo, amp_lo, color="#2c3e50", lw=0.85)
    ax1.axvline(f_orb, color="#c0392b", ls="--", lw=1.2, label=f"$f_{{\\rm orb}} = {f_orb:.4f}\\,\\mathrm{{d}}^{{-1}}$")
    ax1.axvline(f_ps, color="#1e8449", ls="-.", lw=1.3, label=f"Hut (1981) $f_{{\\rm ps}} = {f_ps:.4f}\\,\\mathrm{{d}}^{{-1}}$")
    ax1.axvline(2 * f_orb, color="#c0392b", ls=":", lw=1.0, alpha=0.75, label=r"$2f_{\rm orb}$ (subtracted)")
    ax1.set_xlim(0.20, 1.00)
    ax1.set_xlabel(r"Frequency ($\mathrm{d}^{-1}$)")
    ax1.set_ylabel("Amplitude (ppm)")
    ax1.set_title("Post-$2f_{\\rm orb}$ Residual Amplitude Spectrum", fontweight="bold")
    ax1.legend(fontsize=8.3, loc="upper right")
    ax1.grid(alpha=0.25)

    f_scan = np.linspace(0.28, 0.48, 4000)
    p_rot_s, amp_rot_s = [], []
    for _, grp in tess.groupby("sector"):
        ts = grp["btjd"].values.astype(np.float64)
        fs = grp["flux_rel"].values.astype(np.float64)
        dts = ts - np.mean(ts)
        Xs = np.column_stack([np.ones_like(dts), np.cos(4 * np.pi * f_orb * dts), np.sin(4 * np.pi * f_orb * dts)])
        bs, _, _, _ = np.linalg.lstsq(Xs, fs, rcond=None)
        rs = fs - Xs @ bs
        rs0 = rs - np.mean(rs)
        pgs = AstropyLS(dts, rs0, center_data=True, fit_mean=True).power(f_scan, method="fast", assume_regular_frequency=True)
        amps = 1e6 * np.sqrt(2.0 * np.clip(pgs, 0, 1) * np.var(rs0))
        p_rot_s.append(1.0 / f_scan[int(np.argmax(amps))])
        amp_rot_s.append(float(np.max(amps)))

    sc = ax2.scatter(sdf["t_mid"], p_rot_s, c=amp_rot_s, cmap="viridis", s=78, edgecolor="k", zorder=4)
    ax2.axhline(P_TESS, color="#c0392b", ls="--", lw=1.2, label=f"$P_{{\\rm orb}} = {P_TESS:.3f}$ d")
    ax2.axhline(1.0 / f_ps, color="#1e8449", ls="-.", lw=1.3, label=f"$P_{{\\rm ps}}$ (Hut 1981) $= {1/f_ps:.3f}$ d")
    cbar = plt.colorbar(sc, ax=ax2)
    cbar.set_label("Spot Semi-Amplitude (ppm)")
    ax2.set_xlabel("Sector Midpoint Epoch (BTJD)")
    ax2.set_ylabel("Dominant Spot Period $P_{\\rm rot}$ (d)")
    ax2.set_title("Migrating Near-Synchronous Spot Period (4.6 yr)", fontweight="bold")
    ax2.legend(fontsize=8.3, loc="upper left")
    ax2.grid(alpha=0.25)

    for ext in ("png", "pdf"):
        fig.savefig(f"fig2_frequency_and_spots.{ext}", dpi=300)
    plt.close(fig)
    gc.collect()

    # =========================================================================
    # FIG 3: Optical-to-MIR Photospheric SED + GALEX UV Hot-WD Limit + Residuals
    # =========================================================================
    sed_opt = sed[sed["lam_um"] >= 0.40].copy()
    sed_uv = sed[sed["lam_um"] < 0.40].copy()

    def res_opt(p):
        teff, r_eff, av = p
        ym = planck_lam_fl(sed_opt["lam_um"].values, teff, r_eff, DIST_PC, av, sed_opt["k_av"].values)
        return np.append((sed_opt["lam_Fl_wm2"].values - ym) / sed_opt["lam_Fl_err"].values, (av - AV_GAIA) / 0.05)

    lsq_o = least_squares(res_opt, [6500.0, 2.05, AV_GAIA], bounds=([4500.0, 0.5, 0.0], [9500.0, 6.0, 1.0]))
    t_opt, r_opt, av_opt = lsq_o.x
    ym_opt = planck_lam_fl(sed_opt["lam_um"].values, t_opt, r_opt, DIST_PC, av_opt, sed_opt["k_av"].values)
    chi2_o = float(np.sum(((sed_opt["lam_Fl_wm2"].values - ym_opt) / sed_opt["lam_Fl_err"].values) ** 2))
    dof_o = len(sed_opt) - 3
    err_o = np.sqrt(np.diag((chi2_o / dof_o) * np.linalg.pinv(lsq_o.jac.T @ lsq_o.jac)))
    print(f"[FIG 3] Optical-MIR SED fit (16 bands): Teff={t_opt:.0f}+/-{err_o[0]:.0f} K, R_eff={r_opt:.3f}+/-{err_o[1]:.3f} Rsun, A_V={av_opt:.3f}+/-{err_o[2]:.3f}, chi2/dof={chi2_o:.1f}/{dof_o}")

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.0, 6.0), gridspec_kw={"height_ratios": [3.0, 1.0]}, sharex=True, constrained_layout=True)
    lam_grid = np.logspace(np.log10(0.12), np.log10(15.0), 500)
    k_grid = np.interp(lam_grid, sed["lam_um"].values, sed["k_av"].values)

    y_phot = planck_lam_fl(lam_grid, TEFF_GAIA, R_FLAME, DIST_PC, AV_GAIA, k_grid)
    y_fit = planck_lam_fl(lam_grid, t_opt, r_opt, DIST_PC, av_opt, k_grid)
    # Hot 0.6 Msun WD (T=25,000 K, R=0.012 Rsun) vs Massive 1.2 Msun WD (T=25,000 K, R=0.0055 Rsun)
    y_wd_hot = planck_lam_fl(lam_grid, 25000.0, 0.012, DIST_PC, AV_GAIA, k_grid)

    ax1.loglog(lam_grid, y_phot, color="#c0392b", lw=1.9, label=f"Gaia photosphere ($T_{{\\rm eff}}={TEFF_GAIA:.0f}$ K, $R_{{\\rm eff}}={R_FLAME:.2f}\\,R_\\odot$, $A_V={AV_GAIA:.2f}$)")
    ax1.loglog(lam_grid, y_fit, color="#117a65", ls="--", lw=1.5, label=f"Optical--MIR BB fit ($T={t_opt:.0f}$ K, $R_{{\\rm eff}}={r_opt:.2f}\\,R_\\odot$)")
    ax1.loglog(lam_grid, y_wd_hot, color="#8e44ad", ls=":", lw=1.5, label=r"Reference hot WD ($T=25{,}000$ K, $R=0.012\,R_\odot$)")
    ax1.errorbar(sed_opt["lam_um"], sed_opt["lam_Fl_wm2"], yerr=sed_opt["lam_Fl_err"], fmt="o", ms=5.5, color="#1f4e79", ecolor="#5d6d7e", capsize=2.5, label="Optical--MIR (Tycho, APASS, Gaia, 2MASS, WISE)")
    ax1.errorbar(sed_uv["lam_um"], sed_uv["lam_Fl_wm2"], yerr=sed_uv["lam_Fl_err"], fmt="D", ms=6.0, color="#d35400", ecolor="#d35400", capsize=2.5, label="GALEX FUV & NUV (line-blanketed F5V)")
    ax1.set_ylim(3e-16, 1.2e-12)
    ax1.set_xlim(0.12, 14.5)
    ax1.set_ylabel(r"$\lambda F_\lambda$ ($\mathrm{W\,m^{-2}}$)")
    ax1.set_title("TYC 4517-88-1: 18-Band Spectral Energy Distribution (FUV to Mid-IR)", fontweight="bold")
    ax1.legend(fontsize=7.8, loc="lower left", framealpha=0.92)
    ax1.grid(True, which="both", alpha=0.25)

    # Fractional residuals on optical-MIR bands relative to Gaia photosphere
    y_ref_opt = planck_lam_fl(sed_opt["lam_um"].values, TEFF_GAIA, R_FLAME, DIST_PC, AV_GAIA, sed_opt["k_av"].values)
    frac_res = (sed_opt["lam_Fl_wm2"].values - y_ref_opt) / y_ref_opt
    frac_err = sed_opt["lam_Fl_err"].values / y_ref_opt
    ax2.errorbar(sed_opt["lam_um"], 100.0 * frac_res, yerr=100.0 * frac_err, fmt="o", ms=4.5, color="#1f4e79", capsize=2.0)
    ax2.axhline(0.0, color="#c0392b", ls="--", lw=1.0)
    ax2.set_ylim(-30, 30)
    ax2.set_xlabel(r"Wavelength $\lambda$ ($\mu\mathrm{m}$)")
    ax2.set_ylabel("Opt--MIR\nResid. (%)")
    ax2.grid(True, which="both", alpha=0.25)

    for ext in ("png", "pdf"):
        fig.savefig(f"fig3_sed_and_hr_position.{ext}", dpi=300)
    plt.close(fig)
    gc.collect()

    # =========================================================================
    # FIG 4: 3-Panel Companion Constraints (Ellipsoidal + Beaming Veto + M2 vs i)
    # =========================================================================
    u_ld, tau_gd = 0.35, 0.35
    c_ell = 0.15 * (15.0 + u_ld) / (3.0 - u_ld) * (1.0 + tau_gd)

    # Grid 1: Dark compact companion (M1 = 1.49 Msun, R1 = 2.06 Rsun, L2 = 0)
    inc_dark = np.linspace(18.0, 90.0, 300)
    m2_dark = np.array([solve_m2(FM_MSUN, M_FLAME, i) for i in inc_dark])
    q_dark = m2_dark / M_FLAME
    a_dark = 4.206 * np.power(M_FLAME + m2_dark, 1 / 3) * (P_TESS ** (2 / 3))
    a2_dark = 1e6 * c_ell * q_dark * np.power(R_FLAME / a_dark, 3) * (np.sin(np.radians(inc_dark)) ** 2)
    beam_dark = np.full_like(inc_dark, 831.0)
    idx_dark = int(np.argmin(np.abs(a2_dark - A2_MEAN_PPM)))

    # Grid 2: Self-consistent luminous F+F/G binary (M1 = 1.32 Msun, R_eff = 2.06 Rsun split by flux)
    # Physically allowed domain: i >= 55 deg (where q <= 1.12 so companion does not outshine primary)
    inc_lum = np.linspace(55.0, 90.0, 200)
    m1_lum = 1.32
    m2_lum = np.array([solve_m2(FM_MSUN, m1_lum, i) for i in inc_lum])
    q_lum = m2_lum / m1_lum
    a_lum = 4.206 * np.power(m1_lum + m2_lum, 1 / 3) * (P_TESS ** (2 / 3))
    r_ratio = np.power(q_lum, 0.95)
    t_ratio = np.power(q_lum, 0.55)
    l2_l1 = (r_ratio ** 2) * np.power(t_ratio, 2.5)
    l1_f = 1.0 / (1.0 + l2_l1)
    l2_f = 1.0 - l1_f
    r1_l = R_FLAME * np.sqrt(l1_f)
    r2_l = r1_l * r_ratio
    a2_1 = c_ell * q_lum * np.power(r1_l / a_lum, 3) * (np.sin(np.radians(inc_lum)) ** 2)
    a2_2 = c_ell * (1.0 / q_lum) * np.power(r2_l / a_lum, 3) * (np.sin(np.radians(inc_lum)) ** 2)
    a2_lum = 1e6 * (l1_f * a2_1 + l2_f * a2_2)
    beam_lum = 1e6 * (K1_KMS / 299792.458) * 2.65 * np.abs(l1_f - l2_f / q_lum)
    idx_lum = int(np.argmin(np.abs(a2_lum - A2_MEAN_PPM)))
    i_no_ecl = float(np.degrees(np.arccos((r1_l[idx_lum] + r2_l[idx_lum]) / a_lum[idx_lum])))

    print(
        f"[FIG 4] Dark solution: i={inc_dark[idx_dark]:.1f} deg, M2={m2_dark[idx_dark]:.2f} Msun, Beam=831 ppm | "
        f"Luminous F+F/G solution (M1=1.32): i={inc_lum[idx_lum]:.1f} deg, M2={m2_lum[idx_lum]:.2f} Msun "
        f"(q={q_lum[idx_lum]:.2f}, R1={r1_l[idx_lum]:.2f}, R2={r2_l[idx_lum]:.2f} Rsun, L2/Ltot={l2_f[idx_lum]:.2f}), "
        f"NetBeam={beam_lum[idx_lum]:.0f} ppm, i_ecl={i_no_ecl:.1f} deg"
    )

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14.0, 4.4), constrained_layout=True)

    # Panel A: Ellipsoidal A2 constraint
    ax1.axvspan(18, 55, color="#f2f3f4", alpha=0.85, label="MS companion over-luminous")
    ax1.plot(inc_dark, a2_dark, color="#8e44ad", lw=2.0, label="Dark compact ($L_2=0$)")
    ax1.plot(inc_lum, a2_lum, color="#1f618d", lw=2.2, label="Luminous F+F/G ($M_1=1.32\\,M_\\odot$)")
    ax1.axhspan(A2_MEAN_PPM - A2_STD_PPM, A2_MEAN_PPM + A2_STD_PPM, color="#e74c3c", alpha=0.22, label=f"Observed $A_2 = {A2_MEAN_PPM:.0f}\\pm{A2_STD_PPM:.0f}$ ppm")
    ax1.axhline(A2_MEAN_PPM, color="#c0392b", ls="--", lw=1.1)
    ax1.axvline(i_no_ecl, color="#555555", ls=":", lw=1.2, label=f"No-eclipse limit ($i < {i_no_ecl:.0f}^\\circ$)")
    ax1.set_xlim(18, 90)
    ax1.set_ylim(0, 6300)
    ax1.set_xlabel("Orbital Inclination $i$ (deg)")
    ax1.set_ylabel("Ellipsoidal $A_2$ (ppm)")
    ax1.set_title("(a) Ellipsoidal Tidal Constraint", fontweight="bold")
    ax1.legend(fontsize=7.4, loc="upper left", framealpha=0.92)
    ax1.grid(alpha=0.25)

    # Panel B: Relativistic Doppler Beaming Veto
    ax2.plot(inc_dark, beam_dark, color="#8e44ad", lw=2.0, ls="--", label="Dark compact ($L_2=0$: 831 ppm)")
    ax2.plot(inc_lum, beam_lum, color="#1f618d", lw=2.2, label="Luminous F+F/G (beaming cancels)")
    ax2.axhspan(max(0, A1_BEAM_PROJ_PPM - A1_BEAM_ERR_PPM), A1_BEAM_PROJ_PPM + A1_BEAM_ERR_PPM, color="#117a65", alpha=0.25, label=f"Coherent RV-phase $A_{{1,\\parallel}} = {A1_BEAM_PROJ_PPM:.0f}\\pm{A1_BEAM_ERR_PPM:.0f}$ ppm")
    ax2.axhline(A1_QUIET_FLOOR_PPM, color="#117a65", ls=":", lw=1.2, label="Quiet-sector $A_1$ floor (135--197 ppm)")
    ax2.scatter([inc_dark[idx_dark]], [831.0], color="#8e44ad", s=75, zorder=5)
    ax2.scatter([inc_lum[idx_lum]], [beam_lum[idx_lum]], color="#1f618d", s=75, marker="D", zorder=5)
    ax2.annotate(r"$>5\sigma$ tension" + "\n(rules out dark BH)", xy=(inc_dark[idx_dark], 831.0), xytext=(33, 660),
                 arrowprops=dict(arrowstyle="->", color="#8e44ad", lw=1.2), fontsize=8.0, color="#6c3483", fontweight="bold")
    ax2.set_xlim(18, 90)
    ax2.set_ylim(0, 1020)
    ax2.set_xlabel("Orbital Inclination $i$ (deg)")
    ax2.set_ylabel("Net Doppler Beaming $A_{\\rm beam}$ (ppm)")
    ax2.set_title("(b) Relativistic Beaming Veto", fontweight="bold")
    ax2.legend(fontsize=7.4, loc="center right", framealpha=0.92)
    ax2.grid(alpha=0.25)

    # Panel C: Companion Mass M2 vs Inclination
    ax3.plot(inc_dark, m2_dark, color="#8e44ad", lw=1.8, ls="--", label=r"$M_2(i)$ for $M_1=1.49\,M_\odot$ (dark)")
    ax3.plot(inc_lum, m2_lum, color="#1f618d", lw=2.2, label=r"$M_2(i)$ for $M_1=1.32\,M_\odot$ (F+F/G)")
    ax3.axhline(1.40, color="#d35400", ls=":", lw=1.2, label=r"Chandrasekhar limit ($1.40\,M_\odot$)")
    ax3.scatter([inc_dark[idx_dark]], [m2_dark[idx_dark]], color="#8e44ad", s=75, zorder=5, label=f"Dark impostor: $i={inc_dark[idx_dark]:.0f}^\\circ, M_2={m2_dark[idx_dark]:.2f}\\,M_\\odot$")
    ax3.scatter([inc_lum[idx_lum]], [m2_lum[idx_lum]], color="#1f618d", s=75, marker="D", zorder=5, label=f"Favored F+F/G: $i={inc_lum[idx_lum]:.0f}^\\circ, M_2={m2_lum[idx_lum]:.2f}\\,M_\\odot$")
    ax3.set_xlim(18, 90)
    ax3.set_ylim(0.8, 5.8)
    ax3.set_xlabel("Orbital Inclination $i$ (deg)")
    ax3.set_ylabel(r"Companion Mass $M_2$ ($M_\odot$)")
    ax3.set_title("(c) Companion Mass Solutions", fontweight="bold")
    ax3.legend(fontsize=7.4, loc="upper right", framealpha=0.92)
    ax3.grid(alpha=0.25)

    for ext in ("png", "pdf"):
        fig.savefig(f"fig4_companion_constraints.{ext}", dpi=300)
    plt.close(fig)
    gc.collect()
    print("[DONE] Regenerated crisp publication figures fig1..fig4 (.png and .pdf).")


if __name__ == "__main__":
    main()

[FIG 3] Optical-MIR SED fit (16 bands): Teff=6826+/-343 K, R_eff=2.002+/-0.077 Rsun, A_V=0.232+/-0.140, chi2/dof=103.7/13
[FIG 4] Dark solution: i=25.0 deg, M2=5.01 Msun, Beam=831 ppm | Luminous F+F/G solution (M1=1.32): i=58.9 deg, M2=1.37 Msun (q=1.04, R1=1.41, R2=1.46 Rsun, L2/Ltot=0.53), NetBeam=37 ppm, i_ecl=75.1 deg
[DONE] Regenerated crisp publication figures fig1..fig4 (.png and .pdf).


In [10]:
#!/usr/bin/env python3
"""
Phase 5 (Fixed v3): Publication-Grade WISE IR Finder Chart & NASA SPHEREx DR1 (0.75-5.0 um)
Spectrophotometric Extraction for TYC 4517-88-1 (TIC 397073349 / Gaia DR3 555926586677752960).

Fixes matplotlib mathtext syntax (\le -> \leq, removes \text{} inside $...$).
Reuses cached 'spherex_dr1_tile_tic397073349.fits' if already downloaded.

Outputs:
  - tic397073349_spherex_dr1_spectrum.csv    : Calibrated 100-channel SPHEREx DR1 spectrum
  - fig5_field_allwise_spherex.{png,pdf,eps} : Clean 2-panel WISE + SPHEREx finder chart
  - fig3_sed_with_spherex.{png,pdf,eps}      : Upgraded Fig 3 overlaying SPHEREx DR1 onto 18-band SED
"""

import gc
import io
import os
import re
import html
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from astropy.io import fits
from astropy.io.votable import parse_single_table
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_area
from astropy.coordinates import SkyCoord
import astropy.units as u

warnings.filterwarnings("ignore")

TARGET_RA = 54.61398005
TARGET_DEC = 79.34350269
DIST_PC = 592.73
TEFF_GAIA = 6536.5
R_FLAME = 2.0559
AV_GAIA = 0.197

H_PLANCK = 6.62607015e-34
C_LIGHT = 2.99792458e8
K_BOLTZ = 1.380649e-23
R_SUN_M = 6.957e8
PC_M = 3.085677581e16

JY_PER_MJYSR_ARCSEC2 = 1e6 * (np.pi / (180.0 * 3600.0)) ** 2

GAIA_TAP_ENDPOINTS = [
    "https://gea.esac.esa.int/tap-server/tap/sync",
    "https://gaia.aip.de/tap/sync",
]

SPHEREX_SIA_URL = "https://irsa.ipac.caltech.edu/SIA"


def planck_lam_fl(lam_um, teff, r_rsun, dist_pc, av, k_av):
    lam_m = lam_um * 1e-6
    x = np.clip((H_PLANCK * C_LIGHT) / (lam_m * K_BOLTZ * teff), 1e-6, 700.0)
    b_lam = (2.0 * H_PLANCK * C_LIGHT ** 2 / (lam_m ** 5)) / np.expm1(x)
    return (
        lam_m * np.pi * b_lam
        * (r_rsun * R_SUN_M / (dist_pc * PC_M)) ** 2
        * np.power(10.0, -0.4 * av * k_av)
    )


def fetch_gaia_neighbors(radius_arcsec: float = 95.0) -> pd.DataFrame:
    """Fetch Gaia DR3 neighbors around TYC 4517-88-1 for finder chart overlay."""
    adql = f"""
    SELECT source_id, ra, dec, phot_g_mean_mag, phot_rp_mean_mag,
           DISTANCE(POINT('ICRS', ra, dec), POINT('ICRS', {TARGET_RA}, {TARGET_DEC})) * 3600.0 AS sep_arcsec
    FROM gaiadr3.gaia_source
    WHERE 1 = CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {TARGET_RA}, {TARGET_DEC}, {radius_arcsec / 3600.0}))
    ORDER BY sep_arcsec ASC
    """
    payload = {"REQUEST": "doQuery", "LANG": "ADQL", "FORMAT": "csv", "QUERY": adql.strip()}
    for url in GAIA_TAP_ENDPOINTS:
        try:
            r = requests.post(url, data=payload, timeout=45)
            r.raise_for_status()
            if not r.text.strip().startswith("<"):
                return pd.read_csv(io.StringIO(r.text))
        except Exception:
            continue
    return pd.DataFrame(columns=["source_id", "ra", "dec", "phot_g_mean_mag", "sep_arcsec"])


def fetch_wise_ir_cutout(fov_arcmin: float = 3.5, width_px: int = 420):
    """
    Fetch WCS-aligned WISE IR cutout:
      1. Legacy Survey Viewer full-sky unWISE W1/W2 color JPEG (unwise-neo7 / unwise-neo6)
      2. Failover: NASA SkyView WISE 3.4 um FITS image
    """
    headers = {"User-Agent": "Mozilla/5.0 (AstrophysicsPipeline/5.3)"}
    pixscale_arcsec = (fov_arcmin * 60.0) / width_px

    for layer in ("unwise-neo7", "unwise-neo6"):
        url = (
            f"https://www.legacysurvey.org/viewer/cutout.jpg"
            f"?ra={TARGET_RA:.6f}&dec={TARGET_DEC:.6f}"
            f"&layer={layer}&pixscale={pixscale_arcsec:.4f}&size={width_px}"
        )
        try:
            r = requests.get(url, headers=headers, timeout=45)
            r.raise_for_status()
            if "image" in r.headers.get("Content-Type", "").lower() and len(r.content) > 2000:
                img = plt.imread(io.BytesIO(r.content), format="jpg")
                img = np.flipud(img)
                w = WCS(naxis=2)
                w.wcs.crpix = [width_px / 2.0 + 0.5, width_px / 2.0 + 0.5]
                deg_per_px = pixscale_arcsec / 3600.0
                w.wcs.cdelt = [-deg_per_px, deg_per_px]
                w.wcs.crval = [TARGET_RA, TARGET_DEC]
                w.wcs.ctype = ["RA---TAN", "DEC--TAN"]
                print(f"[OK] Downloaded Legacy Survey {layer} IR color cutout ({img.shape})")
                return img, w, True, "unWISE W1/W2 Color"
        except Exception as exc:
            print(f"[WARN] Legacy Survey {layer} failed: {exc}")

    skyview_url = "https://skyview.gsfc.nasa.gov/current/cgi/runquery.pl"
    params = {
        "Position": f"{TARGET_RA:.6f},{TARGET_DEC:.6f}",
        "Survey": "WISE 3.4",
        "Pixels": "300",
        "Size": f"{fov_arcmin / 60.0:.6f}",
        "Return": "FITS",
    }
    print("[INFO] Falling back to NASA SkyView WISE 3.4 um FITS...")
    r = requests.get(skyview_url, params=params, headers=headers, timeout=60)
    r.raise_for_status()
    with fits.open(io.BytesIO(r.content)) as hdul:
        data = np.asarray(hdul[0].data, dtype=np.float64)
        w = WCS(hdul[0].header)
    print(f"[OK] Downloaded NASA SkyView WISE 3.4 um FITS ({data.shape})")
    return data, w, False, "AllWISE W1 (3.4 $\\mu$m)"


def extract_urls_from_votable_text(xml_text: str):
    """Extract all unique http/https URLs from VOTable XML (unescaping &amp;)."""
    unescaped = html.unescape(xml_text)
    raw_urls = re.findall(r"https?://[^\s<>\"']+", unescaped)
    data_urls = []
    for u_str in raw_urls:
        low = u_str.lower()
        if any(skip in low for skip in ("w3.org", "ivoa.net/xml", "xmlns", "schema")):
            continue
        if u_str not in data_urls:
            data_urls.append(u_str)
    return data_urls


def resolve_spherex_fits_url(sia_xml: str, headers: dict) -> str:
    """Locate the direct SPHEREx DR1 FITS cube URL from an IRSA SIA2 VOTable response."""
    candidate_urls = []
    try:
        tbl = parse_single_table(io.BytesIO(sia_xml.encode("utf-8"))).to_table()
        for col in tbl.colnames:
            for val in tbl[col]:
                s = val.decode("utf-8", errors="ignore") if isinstance(val, (bytes, bytearray)) else str(val)
                s = s.strip()
                if s.startswith("http://") or s.startswith("https://"):
                    if s not in candidate_urls:
                        candidate_urls.append(s)
    except Exception:
        pass

    for u_str in extract_urls_from_votable_text(sia_xml):
        if u_str not in candidate_urls:
            candidate_urls.append(u_str)

    if not candidate_urls:
        raise RuntimeError(f"No data URLs found in IRSA SIA2 response. Snippet:\n{sia_xml[:600]}")

    direct_fits = [u for u in candidate_urls if ".fits" in u.lower() and "datalink" not in u.lower()]
    if direct_fits:
        cube_fits = [u for u in direct_fits if "hips5_cube" in u.lower() or "cube" in u.lower()]
        return cube_fits[0] if cube_fits else direct_fits[0]

    dl_urls = [u for u in candidate_urls if "datalink" in u.lower() or "irsa.ipac.caltech.edu" in u.lower()]
    for dl in dl_urls:
        r_dl = requests.get(dl, headers=headers, timeout=60)
        r_dl.raise_for_status()
        if r_dl.content[:8] == b"SIMPLE  ":
            return dl
        inner_urls = extract_urls_from_votable_text(r_dl.text)
        inner_fits = [u for u in inner_urls if ".fits" in u.lower() and "datalink" not in u.lower()]
        if inner_fits:
            cube_fits = [u for u in inner_fits if "hips5_cube" in u.lower() or "cube" in u.lower()]
            return cube_fits[0] if cube_fits else inner_fits[0]

    raise RuntimeError(f"Could not resolve a direct .fits URL from candidates: {candidate_urls[:5]}")


def download_spherex_dr1_cube(cache_path: str = "spherex_dr1_tile_tic397073349.fits") -> str:
    """Query IRSA SIA2 for the SPHEREx DR1 All-Sky Cube tile covering TYC 4517-88-1."""
    if os.path.exists(cache_path) and os.path.getsize(cache_path) > 10_000_000:
        with open(cache_path, "rb") as fh:
            if fh.read(8) == b"SIMPLE  ":
                print(f"[OK] Using cached SPHEREx DR1 All-Sky Cube: {cache_path}")
                return cache_path

    params = {
        "COLLECTION": "spherex_dr1_cube",
        "POS": f"circle {TARGET_RA:.6f} {TARGET_DEC:.6f} 0.01",
    }
    headers = {"User-Agent": "Mozilla/5.0 (AstrophysicsPipeline/5.3)"}
    print("[INFO] Querying IRSA SIA2 for SPHEREx DR1 All-Sky Cube (DOI: 10.26131/IRSA679)...")
    r = requests.get(SPHEREX_SIA_URL, params=params, headers=headers, timeout=60)
    r.raise_for_status()

    cube_url = resolve_spherex_fits_url(r.text, headers)
    print(f"[INFO] Streaming SPHEREx DR1 FITS Cube:\n       {cube_url}")

    with requests.get(cube_url, headers=headers, stream=True, timeout=300) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        downloaded = 0
        first_chunk = True
        with open(cache_path, "wb") as fh:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    if first_chunk:
                        if not chunk.startswith(b"SIMPLE  "):
                            raise RuntimeError(f"Stream is not a FITS file: {chunk[:200]!r}")
                        first_chunk = False
                    fh.write(chunk)
                    downloaded += len(chunk)
        print(f"[OK] Downloaded {downloaded / 1e6:.1f} MB (expected {total / 1e6:.1f} MB) -> {cache_path}")
    return cache_path


def extract_spherex_cube_spectrum_and_cutout(cube_path: str):
    """Extract 100-channel aperture-photometry spectrum and 1.3-2.2 um cutout from SPHEREx DR1 cube."""
    with fits.open(cube_path, memmap=True) as hdul:
        img_hdu = hdul["IMAGE"] if "IMAGE" in hdul else hdul[0]
        hdr = img_hdu.header
        cube = img_hdu.data
        nhits = hdul["NHITS"].data if "NHITS" in hdul else None
        blank_str = str(hdr.get("BLANKCH", ""))

        wcs_3d = WCS(hdr, fobj=hdul)
        wcs_cel = wcs_3d.celestial

        coord = SkyCoord(ra=TARGET_RA * u.deg, dec=TARGET_DEC * u.deg, frame="icrs")
        x_t, y_t = wcs_cel.world_to_pixel(coord)
        x_t, y_t = float(x_t), float(y_t)
        n_ch, ny, nx = cube.shape

        pix_area_deg2 = float(proj_plane_pixel_area(wcs_cel))
        pix_area_arcsec2 = pix_area_deg2 * (3600.0 ** 2)
        pix_scale_arcsec = np.sqrt(pix_area_arcsec2)

        ch_idx = np.arange(n_ch, dtype=float)
        try:
            spec_world = wcs_3d.spectral.pixel_to_world(ch_idx)
            lam_um = spec_world.to(u.um).value
        except Exception:
            lam_um = None
            for h in hdul:
                if h.is_image is False and h.data is not None:
                    for col in h.columns.names:
                        arr = np.asarray(h.data[col]).ravel()
                        if len(arr) == n_ch and np.nanmax(arr) < 1e-4:
                            lam_um = arr * 1e6
                            break
                        if len(arr) == n_ch and 0.5 < np.nanmin(arr) and np.nanmax(arr) < 6.0:
                            lam_um = arr
                            break
            if lam_um is None:
                raise RuntimeError("Could not extract wavelength array from SPHEREx DR1 cube.")

        half = 12
        ix0, iy0 = int(round(x_t)), int(round(y_t))
        x_lo, x_hi = max(0, ix0 - half), min(nx, ix0 + half + 1)
        y_lo, y_hi = max(0, iy0 - half), min(ny, iy0 + half + 1)

        stamp = np.asarray(cube[:, y_lo:y_hi, x_lo:x_hi], dtype=np.float64)
        nh_stamp = np.asarray(nhits[:, y_lo:y_hi, x_lo:x_hi], dtype=np.float64) if nhits is not None else None

        yy, xx = np.mgrid[y_lo:y_hi, x_lo:x_hi]
        rr = np.hypot(xx - x_t, yy - y_t)

        ap_mask = rr <= 2.0
        sky_mask = (rr >= 3.5) & (rr <= 6.5)
        n_ap = int(np.sum(ap_mask))

        blank_set = {int(tok) for tok in blank_str.replace(",", " ").split() if tok.isdigit()}

        spec_rows = []
        for k in range(n_ch):
            ch_1based = k + 1
            if ch_1based in blank_set:
                continue
            plane = stamp[k]
            ap_vals = plane[ap_mask]
            sky_vals = plane[sky_mask]
            good_ap = np.isfinite(ap_vals)
            good_sky = np.isfinite(sky_vals)
            if np.sum(good_ap) < max(5, int(0.75 * n_ap)) or np.sum(good_sky) < 15:
                continue

            if nh_stamp is not None:
                nh_center = nh_stamp[k, iy0 - y_lo, ix0 - x_lo]
                if not np.isfinite(nh_center) or nh_center <= 0:
                    continue

            sky_med = float(np.nanmedian(sky_vals[good_sky]))
            sky_mad = 1.4826 * float(np.nanmedian(np.abs(sky_vals[good_sky] - sky_med)))
            sky_clean = sky_vals[good_sky]
            if sky_mad > 0:
                sky_clean = sky_clean[np.abs(sky_clean - sky_med) < 3.0 * sky_mad]
            if len(sky_clean) < 10:
                continue
            sky_bg = float(np.median(sky_clean))
            sky_std = 1.4826 * float(np.median(np.abs(sky_clean - sky_bg)))

            net_sum_mjysr = float(np.sum(ap_vals[good_ap] - sky_bg))
            if np.sum(good_ap) < n_ap:
                net_sum_mjysr *= (n_ap / np.sum(good_ap))

            fnu_jy = net_sum_mjysr * JY_PER_MJYSR_ARCSEC2 * pix_area_arcsec2
            err_sum_mjysr = sky_std * np.sqrt(n_ap * (1.0 + n_ap / len(sky_clean)))
            fnu_err_jy = err_sum_mjysr * JY_PER_MJYSR_ARCSEC2 * pix_area_arcsec2

            if fnu_jy <= 0:
                continue

            lam_val = float(lam_um[k])
            nu_hz = C_LIGHT / (lam_val * 1e-6)
            lam_fl = nu_hz * fnu_jy * 1e-26
            lam_fl_err = nu_hz * fnu_err_jy * 1e-26

            spec_rows.append({
                "channel": ch_1based,
                "lam_um": lam_val,
                "fnu_mjy": fnu_jy * 1e3,
                "fnu_err_mjy": fnu_err_jy * 1e3,
                "lam_Fl_wm2": lam_fl,
                "lam_Fl_err": lam_fl_err,
                "snr": fnu_jy / max(fnu_err_jy, 1e-12),
            })

        spec_df = pd.DataFrame(spec_rows).sort_values("lam_um").reset_index(drop=True)
        b23_mask = (lam_um >= 1.30) & (lam_um <= 2.20)
        b23_img = np.nanmedian(stamp[b23_mask], axis=0)
        wcs_stamp = wcs_cel.slice((slice(y_lo, y_hi), slice(x_lo, x_hi)))

    return spec_df, b23_img, wcs_stamp, pix_scale_arcsec


def main():
    print("=" * 96)
    print("PHASE 5: WISE IR FINDER CHART & NASA SPHEREx DR1 (0.75-5.0 um) SPECTROPHOTOMETRY")
    print("=" * 96)

    sed = pd.read_csv("tic397073349_sed_photometry.csv")
    nb_df = fetch_gaia_neighbors(radius_arcsec=95.0)

    wise_img, wise_wcs, wise_is_rgb, wise_label = fetch_wise_ir_cutout(fov_arcmin=3.5, width_px=420)
    cube_path = download_spherex_dr1_cube()
    sphx_df, sphx_img, sphx_wcs, sphx_pix = extract_spherex_cube_spectrum_and_cutout(cube_path)

    sphx_clean = sphx_df[sphx_df["snr"] >= 3.0].copy().reset_index(drop=True)
    print(f"[OK] Extracted {len(sphx_clean)} valid SPHEREx DR1 channels ({sphx_clean['lam_um'].min():.3f} - {sphx_clean['lam_um'].max():.3f} um)")
    print(f"     Median channel SNR = {sphx_clean['snr'].median():.1f} | Range = {sphx_clean['snr'].min():.1f} - {sphx_clean['snr'].max():.1f}")

    ref_bands = [
        ("2MASS_J",  1.235, 83.96),
        ("2MASS_H",  1.662, 65.81),
        ("2MASS_Ks", 2.159, 44.87),
        ("WISE_W1",  3.353, 21.49),
        ("WISE_W2",  4.603, 11.74),
    ]
    ratios = []
    for bname, lam_ref, f_ref_mjy in ref_bands:
        sub = sphx_clean[np.abs(sphx_clean["lam_um"] - lam_ref) <= 0.12]
        if len(sub) > 0:
            ratios.append(float(np.median(sub["fnu_mjy"])) / f_ref_mjy)

    ap_corr = 1.0 / np.median(ratios) if ratios else 1.0
    sphx_clean["lam_Fl_corr_wm2"] = sphx_clean["lam_Fl_wm2"] * ap_corr
    sphx_clean["lam_Fl_corr_err"] = sphx_clean["lam_Fl_err"] * ap_corr
    sphx_clean.to_csv("tic397073349_spherex_dr1_spectrum.csv", index=False)

    k_sphx = np.interp(sphx_clean["lam_um"].values, sed["lam_um"].values, sed["k_av"].values)
    y_model_sphx = planck_lam_fl(sphx_clean["lam_um"].values, TEFF_GAIA, R_FLAME, DIST_PC, AV_GAIA, k_sphx)
    rel_dev = 100.0 * (sphx_clean["lam_Fl_corr_wm2"].values - y_model_sphx) / y_model_sphx
    mad_dev = 1.4826 * np.median(np.abs(rel_dev - np.median(rel_dev)))
    print(f"     Aperture-corrected SPHEREx DR1 vs. 6536 K photosphere: median = {np.median(rel_dev):+.1f}%, MAD = {mad_dev:.1f}%")

    # =========================================================================
    # FIG 5: PUBLICATION 2-PANEL FINDER CHART (unWISE IR + SPHEREx DR1 1.3-2.2 um)
    # =========================================================================
    plt.rcParams.update({"font.size": 10, "axes.linewidth": 1.0, "font.family": "serif"})
    fig = plt.figure(figsize=(10.5, 4.8), constrained_layout=True)

    ax1 = fig.add_subplot(1, 2, 1, projection=wise_wcs)
    if wise_is_rgb:
        ax1.imshow(wise_img, origin="lower")
    else:
        v0 = np.nanpercentile(wise_img, 10)
        v1 = np.nanpercentile(wise_img, 99.5)
        ax1.imshow(np.arcsinh((wise_img - v0) / max(v1 - v0, 1e-6) * 10.0), origin="lower", cmap="magma")

    tx, ty = wise_wcs.world_to_pixel(SkyCoord(TARGET_RA * u.deg, TARGET_DEC * u.deg, frame="icrs"))
    pix_scale_w_arcsec = np.sqrt(float(proj_plane_pixel_area(wise_wcs))) * 3600.0
    r42_px = 42.0 / pix_scale_w_arcsec

    ax1.add_patch(Circle((tx, ty), r42_px, edgecolor="#00e5ff", facecolor="none", lw=1.6, ls="--",
                         label=r"TESS 2-pix radius ($42''$)"))
    ax1.plot([tx - 18, tx - 6], [ty, ty], color="yellow", lw=1.4)
    ax1.plot([tx + 6, tx + 18], [ty, ty], color="yellow", lw=1.4)
    ax1.plot([tx, tx], [ty - 18, ty - 6], color="yellow", lw=1.4)
    ax1.plot([tx, tx], [ty + 6, ty + 18], color="yellow", lw=1.4)

    nb_in = nb_df[(nb_df["sep_arcsec"] > 0.5) & (nb_df["sep_arcsec"] <= 42.0)]
    if len(nb_in) > 0:
        nx_p, ny_p = wise_wcs.world_to_pixel(SkyCoord(nb_in["ra"].values * u.deg, nb_in["dec"].values * u.deg, frame="icrs"))
        ax1.scatter(nx_p, ny_p, s=42, facecolors="none", edgecolors="#ff5252", lw=1.1,
                    label=f"Gaia DR3 neighbors $\\leq 42''$ ($N={len(nb_in)}$)")

    ax1.set_xlabel("Right Ascension (ICRS)")
    ax1.set_ylabel("Declination (ICRS)")
    ax1.set_title(f"(a) {wise_label} ($3.5' \\times 3.5'$)", fontweight="bold")
    ax1.legend(fontsize=8.0, loc="upper right", framealpha=0.88)

    ax2 = fig.add_subplot(1, 2, 2, projection=sphx_wcs)
    vmin = np.nanpercentile(sphx_img, 10)
    vmax = np.nanpercentile(sphx_img, 99.5)
    ax2.imshow(np.arcsinh((sphx_img - vmin) / max(vmax - vmin, 1e-6) * 10.0), origin="lower", cmap="inferno")
    sx, sy = sphx_wcs.world_to_pixel(SkyCoord(TARGET_RA * u.deg, TARGET_DEC * u.deg, frame="icrs"))
    r_ap_px = 2.0
    r_tess_spx = 42.0 / sphx_pix
    ax2.add_patch(Circle((sx, sy), r_ap_px, edgecolor="#00e676", facecolor="none", lw=1.5,
                         label=f"SPHEREx aperture ($r={r_ap_px * sphx_pix:.0f}''$)"))
    ax2.add_patch(Circle((sx, sy), r_tess_spx, edgecolor="#00e5ff", facecolor="none", lw=1.4, ls="--",
                         label=r"TESS 2-pix radius ($42''$)"))
    ax2.set_xlabel("Right Ascension (ICRS)")
    ax2.set_ylabel("Declination (ICRS)")
    ax2.set_title(r"(b) NASA SPHEREx DR1 (1.3-2.2 $\mu$m Cube)", fontweight="bold")
    ax2.legend(fontsize=8.0, loc="upper right", framealpha=0.88)

    for ext in ("png", "pdf", "eps"):
        fig.savefig(f"fig5_field_allwise_spherex.{ext}", dpi=300)
    plt.close(fig)
    gc.collect()

    # =========================================================================
    # UPGRADED FIG 3: 18-Band SED + 100-Channel NASA SPHEREx DR1 Spectrum
    # =========================================================================
    sed_opt = sed[sed["lam_um"] >= 0.40].copy()
    sed_uv = sed[sed["lam_um"] < 0.40].copy()

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(8.2, 6.2), gridspec_kw={"height_ratios": [3.0, 1.0]}, sharex=True, constrained_layout=True
    )
    lam_grid = np.logspace(np.log10(0.12), np.log10(15.0), 500)
    k_grid = np.interp(lam_grid, sed["lam_um"].values, sed["k_av"].values)
    y_phot = planck_lam_fl(lam_grid, TEFF_GAIA, R_FLAME, DIST_PC, AV_GAIA, k_grid)
    y_wd_hot = planck_lam_fl(lam_grid, 25000.0, 0.012, DIST_PC, AV_GAIA, k_grid)

    ax1.loglog(lam_grid, y_phot, color="#c0392b", lw=1.8, zorder=3,
               label=f"Gaia photosphere ($T_{{\\rm eff}}={TEFF_GAIA:.0f}$ K, $R_{{\\rm eff}}={R_FLAME:.2f}\\,R_\\odot$, $A_V={AV_GAIA:.2f}$)")
    ax1.loglog(lam_grid, y_wd_hot, color="#8e44ad", ls=":", lw=1.4, zorder=2,
               label=r"Reference hot WD ($T=25{,}000$ K, $R=0.012\,R_\odot$)")
    ax1.errorbar(sphx_clean["lam_um"], sphx_clean["lam_Fl_corr_wm2"], yerr=sphx_clean["lam_Fl_corr_err"],
                 fmt="-", lw=1.2, color="#16a085", ecolor="#a3e4d7", elinewidth=0.7, alpha=0.95, zorder=4,
                 label=f"NASA SPHEREx DR1 cube ({len(sphx_clean)} ch., 0.75-5.0 $\\mu$m)")
    ax1.errorbar(sed_opt["lam_um"], sed_opt["lam_Fl_wm2"], yerr=sed_opt["lam_Fl_err"],
                 fmt="o", ms=5.0, color="#1f4e79", ecolor="#5d6d7e", capsize=2.2, zorder=5,
                 label="Archival Optical-MIR (Tycho, APASS, Gaia, 2MASS, WISE)")
    ax1.errorbar(sed_uv["lam_um"], sed_uv["lam_Fl_wm2"], yerr=sed_uv["lam_Fl_err"],
                 fmt="D", ms=5.8, color="#d35400", ecolor="#d35400", capsize=2.2, zorder=5,
                 label="GALEX FUV & NUV (line-blanketed F5V)")

    ax1.set_ylim(3e-16, 1.3e-12)
    ax1.set_xlim(0.12, 14.5)
    ax1.set_ylabel(r"$\lambda F_\lambda$ ($\mathrm{W\,m^{-2}}$)")
    ax1.set_title(r"TYC 4517-88-1: 18-Band SED + NASA SPHEREx DR1 (0.75-5.0 $\mu$m) Spectrum", fontweight="bold")
    ax1.legend(fontsize=7.6, loc="lower center", framealpha=0.94)
    ax1.grid(True, which="both", alpha=0.25)

    y_ref_opt = planck_lam_fl(sed_opt["lam_um"].values, TEFF_GAIA, R_FLAME, DIST_PC, AV_GAIA, sed_opt["k_av"].values)
    ax2.errorbar(sed_opt["lam_um"], 100.0 * (sed_opt["lam_Fl_wm2"].values - y_ref_opt) / y_ref_opt,
                 yerr=100.0 * sed_opt["lam_Fl_err"].values / y_ref_opt,
                 fmt="o", ms=4.2, color="#1f4e79", capsize=1.8, zorder=4)
    ax2.plot(sphx_clean["lam_um"], rel_dev, ".", ms=3.5, color="#16a085", alpha=0.85, zorder=3)
    ax2.axhline(0.0, color="#c0392b", ls="--", lw=1.0)
    ax2.set_ylim(-30, 30)
    ax2.set_xlabel(r"Wavelength $\lambda$ ($\mu\mathrm{m}$)")
    ax2.set_ylabel("Resid. (%)")
    ax2.grid(True, which="both", alpha=0.25)

    for ext in ("png", "pdf", "eps"):
        fig.savefig(f"fig3_sed_with_spherex.{ext}", dpi=300)
    plt.close(fig)
    gc.collect()
    print("[DONE] Saved tic397073349_spherex_dr1_spectrum.csv, fig5_field_allwise_spherex.*, and fig3_sed_with_spherex.*")


if __name__ == "__main__":
    main()

PHASE 5: WISE IR FINDER CHART & NASA SPHEREx DR1 (0.75-5.0 um) SPECTROPHOTOMETRY
[OK] Downloaded Legacy Survey unwise-neo7 IR color cutout ((420, 420, 3))
[OK] Using cached SPHEREx DR1 All-Sky Cube: spherex_dr1_tile_tic397073349.fits
[OK] Extracted 100 valid SPHEREx DR1 channels (0.753 - 4.988 um)
     Median channel SNR = 413.2 | Range = 41.7 - 1187.4
     Aperture-corrected SPHEREx DR1 vs. 6536 K photosphere: median = +3.0%, MAD = 9.3%


[DONE] Saved tic397073349_spherex_dr1_spectrum.csv, fig5_field_allwise_spherex.*, and fig3_sed_with_spherex.*


In [11]:
#!/usr/bin/env python3
"""
Standalone 3-second fix for fig5_field_allwise_spherex.{png,pdf,eps}:
Binds RA/Dec axis labels directly to WCSAxes coords[0]/coords[1] so the
rotated HEALPix SPHEREx DR1 cube (Panel b) displays accurate RA and Dec labels.
"""

import io
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from astropy.io import fits
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_area
from astropy.coordinates import SkyCoord
import astropy.units as u

warnings.filterwarnings("ignore")

TARGET_RA = 54.61398005
TARGET_DEC = 79.34350269
CUBE_PATH = "spherex_dr1_tile_tic397073349.fits"

GAIA_TAP_ENDPOINTS = [
    "https://gea.esac.esa.int/tap-server/tap/sync",
    "https://gaia.aip.de/tap/sync",
]


def fetch_gaia_neighbors(radius_arcsec: float = 95.0) -> pd.DataFrame:
    adql = f"""
    SELECT source_id, ra, dec, phot_g_mean_mag,
           DISTANCE(POINT('ICRS', ra, dec), POINT('ICRS', {TARGET_RA}, {TARGET_DEC})) * 3600.0 AS sep_arcsec
    FROM gaiadr3.gaia_source
    WHERE 1 = CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {TARGET_RA}, {TARGET_DEC}, {radius_arcsec / 3600.0}))
    ORDER BY sep_arcsec ASC
    """
    payload = {"REQUEST": "doQuery", "LANG": "ADQL", "FORMAT": "csv", "QUERY": adql.strip()}
    for url in GAIA_TAP_ENDPOINTS:
        try:
            r = requests.post(url, data=payload, timeout=45)
            r.raise_for_status()
            if not r.text.strip().startswith("<"):
                return pd.read_csv(io.StringIO(r.text))
        except Exception:
            continue
    return pd.DataFrame(columns=["source_id", "ra", "dec", "phot_g_mean_mag", "sep_arcsec"])


def fetch_unwise_cutout(fov_arcmin: float = 3.5, width_px: int = 420):
    pixscale_arcsec = (fov_arcmin * 60.0) / width_px
    url = (
        f"https://www.legacysurvey.org/viewer/cutout.jpg"
        f"?ra={TARGET_RA:.6f}&dec={TARGET_DEC:.6f}"
        f"&layer=unwise-neo7&pixscale={pixscale_arcsec:.4f}&size={width_px}"
    )
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=45)
    r.raise_for_status()
    img = np.flipud(plt.imread(io.BytesIO(r.content), format="jpg"))
    w = WCS(naxis=2)
    w.wcs.crpix = [width_px / 2.0 + 0.5, width_px / 2.0 + 0.5]
    deg_per_px = pixscale_arcsec / 3600.0
    w.wcs.cdelt = [-deg_per_px, deg_per_px]
    w.wcs.crval = [TARGET_RA, TARGET_DEC]
    w.wcs.ctype = ["RA---TAN", "DEC--TAN"]
    return img, w


def load_spherex_cutout(cube_path: str):
    with fits.open(cube_path, memmap=True) as hdul:
        img_hdu = hdul["IMAGE"]
        cube = img_hdu.data
        wcs_3d = WCS(img_hdu.header, fobj=hdul)
        wcs_cel = wcs_3d.celestial
        lam_um = wcs_3d.spectral.pixel_to_world(np.arange(cube.shape[0], dtype=float)).to(u.um).value
        x_t, y_t = wcs_cel.world_to_pixel(SkyCoord(TARGET_RA * u.deg, TARGET_DEC * u.deg, frame="icrs"))
        ix0, iy0 = int(round(float(x_t))), int(round(float(y_t)))
        half = 12
        y_lo, y_hi = max(0, iy0 - half), min(cube.shape[1], iy0 + half + 1)
        x_lo, x_hi = max(0, ix0 - half), min(cube.shape[2], ix0 + half + 1)
        b23_mask = (lam_um >= 1.30) & (lam_um <= 2.20)
        b23_img = np.nanmedian(np.asarray(cube[b23_mask, y_lo:y_hi, x_lo:x_hi], dtype=np.float64), axis=0)
        wcs_stamp = wcs_cel.slice((slice(y_lo, y_hi), slice(x_lo, x_hi)))
        pix_scale_arcsec = np.sqrt(float(proj_plane_pixel_area(wcs_cel))) * 3600.0
    return b23_img, wcs_stamp, pix_scale_arcsec


def main():
    nb_df = fetch_gaia_neighbors(95.0)
    wise_img, wise_wcs = fetch_unwise_cutout(3.5, 420)
    sphx_img, sphx_wcs, sphx_pix = load_spherex_cutout(CUBE_PATH)

    plt.rcParams.update({"font.size": 10, "axes.linewidth": 1.0, "font.family": "serif"})
    fig = plt.figure(figsize=(10.5, 4.8), constrained_layout=True)

    # Panel (a): unWISE W1/W2 Color (North-up TAN projection)
    ax1 = fig.add_subplot(1, 2, 1, projection=wise_wcs)
    ax1.imshow(wise_img, origin="lower")
    tx, ty = wise_wcs.world_to_pixel(SkyCoord(TARGET_RA * u.deg, TARGET_DEC * u.deg, frame="icrs"))
    pix_w = np.sqrt(float(proj_plane_pixel_area(wise_wcs))) * 3600.0
    ax1.add_patch(Circle((tx, ty), 42.0 / pix_w, edgecolor="#00e5ff", facecolor="none", lw=1.6, ls="--",
                         label=r"TESS 2-pix radius ($42''$)"))
    ax1.plot([tx - 18, tx - 6], [ty, ty], color="yellow", lw=1.4)
    ax1.plot([tx + 6, tx + 18], [ty, ty], color="yellow", lw=1.4)
    ax1.plot([tx, tx], [ty - 18, ty - 6], color="yellow", lw=1.4)
    ax1.plot([tx, tx], [ty + 6, ty + 18], color="yellow", lw=1.4)

    nb_in = nb_df[(nb_df["sep_arcsec"] > 0.5) & (nb_df["sep_arcsec"] <= 42.0)]
    if len(nb_in) > 0:
        nx_p, ny_p = wise_wcs.world_to_pixel(SkyCoord(nb_in["ra"].values * u.deg, nb_in["dec"].values * u.deg, frame="icrs"))
        ax1.scatter(nx_p, ny_p, s=42, facecolors="none", edgecolors="#ff5252", lw=1.1,
                    label=f"Gaia DR3 neighbors $\\leq 42''$ ($N={len(nb_in)}$)")

    ax1.coords[0].set_axislabel("Right Ascension (ICRS)")
    ax1.coords[1].set_axislabel("Declination (ICRS)")
    ax1.set_title(r"(a) unWISE W1/W2 Color ($3.5' \times 3.5'$)", fontweight="bold")
    ax1.legend(fontsize=8.0, loc="upper right", framealpha=0.88)

    # Panel (b): SPHEREx DR1 1.3-2.2 um Cube (HEALPix projection -> WCSAxes auto-maps RA/Dec)
    ax2 = fig.add_subplot(1, 2, 2, projection=sphx_wcs)
    vmin, vmax = np.nanpercentile(sphx_img, 10), np.nanpercentile(sphx_img, 99.5)
    ax2.imshow(np.arcsinh((sphx_img - vmin) / max(vmax - vmin, 1e-6) * 10.0), origin="lower", cmap="inferno")
    sx, sy = sphx_wcs.world_to_pixel(SkyCoord(TARGET_RA * u.deg, TARGET_DEC * u.deg, frame="icrs"))
    ax2.add_patch(Circle((sx, sy), 2.0, edgecolor="#00e676", facecolor="none", lw=1.5,
                         label=f"SPHEREx aperture ($r={2.0 * sphx_pix:.0f}''$)"))
    ax2.add_patch(Circle((sx, sy), 42.0 / sphx_pix, edgecolor="#00e5ff", facecolor="none", lw=1.4, ls="--",
                         label=r"TESS 2-pix radius ($42''$)"))
    ax2.coords[0].set_axislabel("Right Ascension (ICRS)")
    ax2.coords[1].set_axislabel("Declination (ICRS)")
    ax2.set_title(r"(b) NASA SPHEREx DR1 (1.3-2.2 $\mu$m Cube)", fontweight="bold")
    ax2.legend(fontsize=8.0, loc="upper right", framealpha=0.88)

    for ext in ("png", "pdf", "eps"):
        fig.savefig(f"fig5_field_allwise_spherex.{ext}", dpi=300)
    plt.close(fig)
    print("[OK] Regenerated fig5_field_allwise_spherex.{png,pdf,eps} with corrected HEALPix RA/Dec labels.")


if __name__ == "__main__":
    main()

[OK] Regenerated fig5_field_allwise_spherex.{png,pdf,eps} with corrected HEALPix RA/Dec labels.
